# 03 — Análise exploratória
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** conhece cada variável sozinha (seção 5), mede como as variáveis se relacionam entre si
(seção 6) e com o churn (seção 7), dá um veredito para cada uma (seção 8), grava o dicionário completo (seção 9) e
calcula o valor do churn e a linha de base que o modelo precisa superar (seção 10).

**Por que leio o dado como veio, e não a tabela limpa:** os valores inválidos ficam de fora dos cálculos, e nenhuma
conclusão depende do que a limpeza preencheu. Leio das tabelas **bronze**, a fonte única do grupo (D17). A tabela limpa
(silver, notebook 02) é a que vai para o modelo.

A teoria de cada teste está no **apêndice**, no fim do notebook.

| Notebook | Papel | Seções |
|---|---|---|
| 00 — Documentação | O projeto de ponta a ponta, decisões e perguntas em aberto | — |
| `_comum` | Leitura dos dados, padrão dos gráficos, metadados e verificação de qualidade, usados por todos os notebooks | — |
| 01 — Dicionário de dados | O que é cada coluna, o que está errado e o que levar ao grupo | 1 a 4 |
| 02 — Preparação dos dados | Aplica as decisões do 01 e grava as tabelas limpas | — |
| 03 — Análise exploratória | Descritiva, associações, churn, vazamento e avaliação de cada variável | 5 a 10 |
| 04 — Variáveis derivadas e ABT | Variáveis derivadas, ABT e dicionário com a sugestão de cada coluna; grava a gold | — |
| 05 — Seleção de variáveis | Separa treino e teste, faz os testes estatísticos e chega às 41 variáveis do modelo; confere com os métodos da aula | — |
| 06 — Modelagem | Compara 6 modelos, escolhe o modelo e o limiar, publica as previsões e registra no MLflow | — |
| 07 — Segmentação | Agrupa os clientes por comportamento (K-means) e cruza com risco e valor | — |
| 08 — Entrega | Gera o `prediction` da entrega, confere o formato e resume os números para a apresentação | — |

A numeração das seções continua entre o 01 e o 03, para as referências cruzadas (por exemplo, "4.3" ou "7.3") valerem.

## Checklist de decisões

**A mesma tabela em todos os notebooks.** Este notebook traz a evidência da D9 (seção 7.3), da D10 (seção 6), da D12 e da
D15 (seções 7 e 8). A D16 fica para o 04. ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

## Glossário de siglas

Toda sigla usada nos notebooks, por extenso, com o lugar onde ela é explicada em detalhe.

| Sigla | Por extenso | O que é, em uma frase | Onde entender melhor |
|---|---|---|---|
| **IV** | *Information Value* (valor da informação) | Quanto uma variável, sozinha, separa quem sai de quem fica (0 = nada; acima de 0,5 = muito forte) | 03: seção 7.1 e Apêndice A.7 · 04: seção 1 (exemplo calculado) |
| **WoE** | *Weight of Evidence* (peso da evidência) | Em cada grupo, ln(% dos que saíram ÷ % dos que ficaram); positivo = risco acima da média. É a base do IV | 03: Apêndice A.7 · 04: seção 1 |
| **PSI** | *Population Stability Index* (índice de estabilidade da população) | Quanto a distribuição de uma variável mudou entre o histórico e a inferência (abaixo de 0,10 = estável) | 03: seção 5.9 (com exemplo e cenário inventado) |
| **ABT** | *Analytical Base Table* (tabela analítica de base) | A tabela que o modelo lê: uma linha por cliente, uma coluna por variável, tudo limpo | 04: seção 2 |
| **KS** | Teste de Kolmogorov-Smirnov | Testa se duas distribuições de números são iguais | 03: seção 5.9 |
| **CV** | Coeficiente de variação | Desvio-padrão ÷ média, em %: quanto os valores variam em relação ao tamanho deles | 03: seção 5.1 |
| **Q1, Q3, IQR** | Primeiro e terceiro quartis; *interquartile range* (amplitude interquartil) | Q1 e Q3 deixam 25% dos valores abaixo e acima; IQR = Q3 − Q1, a largura da caixa do boxplot | 03: seção 5.3 (critério de Tukey) |
| **V** | V de Cramér | Associação entre duas variáveis de categorias, de 0 a 1 | 03: seção 6 e Apêndice A.1 · nota "Associação entre variáveis" |
| **ρ** (rô) | Correlação de Spearman | Associação entre duas variáveis numéricas, pela ordem dos valores | 03: Apêndice A.2 |
| **η** (eta) | Razão de correlação | Quanto uma variável de categorias explica de uma numérica | 03: Apêndice A.3 |
| **r** | Correlação rank-bisserial | Tamanho do efeito do teste de Mann-Whitney (quem saiu × quem ficou) | 03: Apêndice A.6 |
| **p** | p-valor | A chance de ver uma diferença desse tamanho só por acaso; abaixo de 0,05, dificilmente é acaso | 03: Apêndice A.1 |
| **D1 a D18** | Decisões do checklist | Cada decisão do grupo, com o status (☑ decidido, ◐ sugestão, ☐ em aberto) | Tabela "Checklist de decisões" no início de cada notebook |
| **bronze, silver, gold** | Camadas da arquitetura medalhão | Dado bruto (bronze), limpo (silver) e pronto para o modelo (gold) | 01: seção 2 · 02 e 04 |

In [ ]:
%run ./_comum

## Fonte: tabelas bronze (D17)

O mesmo dado dos CSVs, como chegou, lido da fonte única do grupo. As colunas de controle saem aqui.

In [ ]:
hist_raw = spark.table(f"{FQ}.bronze_history").toPandas().drop(columns=["_origem", "_carregado_em"])
inf_raw = spark.table(f"{FQ}.bronze_inference").toPandas().drop(columns=["_origem", "_carregado_em"])
assert (len(hist_raw), len(inf_raw)) == (5634, 1409), "a bronze não tem o número de clientes esperado: rodar o notebook 01"
print(f"bronze_history: {virg(len(hist_raw), 0)} clientes · bronze_inference: {virg(len(inf_raw), 0)} clientes")

## 5. Estatística descritiva: conhecer cada variável sozinha

**O que faço:** resumo cada variável com números e gráficos, **uma de cada vez** (análise univariada).
As relações entre as variáveis vêm na seção 6, e a relação com o churn, na seção 7.

**Por quê?**
- Antes de perguntar "o que explica o churn", preciso saber como cada variável se comporta: onde está o centro,
  quanto ela varia, se é simétrica, se tem valores extremos.
- O resumo depende do tipo (seção 3):
  - **quantitativas:** posição (média, mediana, quartis), dispersão (desvio-padrão, coeficiente de variação), forma
    (assimetria, curtose) e valores extremos (critério de Tukey), com histograma e boxplot;
  - **qualitativas:** frequência e percentual de cada categoria, com gráfico de barras.
- Deixo **fora** dos cálculos os valores inválidos encontrados na seção 4 (`unknown`, `null`), e o número de valores
  válidos (`n_validos`) aparece em cada estatística.

> **Resumo: as medidas que uso**
> - **Posição:** média e mediana. Quando a distribuição é assimétrica ou tem dois picos, a mediana representa melhor o centro.
> - **Coeficiente de variação (CV)** = desvio-padrão ÷ média. Compara a dispersão de variáveis com unidades diferentes.
>   Acima de 30%, a variável é considerada muito dispersa.
> - **Assimetria:** 0 = simétrica; positiva = cauda longa à direita (poucos valores altos); negativa = cauda à esquerda.
> - **Curtose (excesso):** 0 = como a normal; negativa = distribuição achatada, com valores espalhados;
>   positiva = pico alto e caudas pesadas.
> - **Valores extremos:** critério de Tukey, explicado antes do boxplot (5.4).

In [ ]:
# 5.2 Estatística descritiva das variáveis quantitativas (valores válidos; inválidos ficam de fora)
from scipy import stats

QUANT = meta.index[meta["tipo_estatistico"].str.startswith("quantitativa")].tolist()

def descrever(x: pd.Series) -> dict:
    x = pd.to_numeric(x, errors="coerce").dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    return {"n_validos": len(x), "media": x.mean(), "desvio_padrao": x.std(), "cv_pct": 100 * x.std() / x.mean(),
            "minimo": x.min(), "q1": q1, "mediana": x.median(), "q3": q3, "maximo": x.max(),
            "assimetria": stats.skew(x), "curtose": stats.kurtosis(x)}

desc_num = pd.DataFrame({col: descrever(hist_raw[col]) for col in QUANT}).T
desc_num.index.name = "variavel"
display(desc_num.round(2).reset_index())

In [ ]:
# 5.3 Histograma de cada variável quantitativa: em azul, as faixas que o título cita
BINS = {
    "tenure": range(0, 78, 6),              # semestres
    "MonthlyCharges": range(10, 130, 10),   # faixas de R$ 10
    "TotalCharges": range(0, 10000, 1000),  # faixas de R$ 1.000
    "MonthlyIncome": range(0, 11000, 1000), # faixas de 1.000
}
DESTAQUE_FAIXAS = {                         # faixas (início, fim) que o título cita, com o rótulo de cada uma
    "tenure": [((0, 6), "1º semestre"), ((66, 72), "66 a 72 meses")],
    "MonthlyCharges": [((20, 30), "só telefonia"), ((70, 110), "pacotes com internet")],
    "TotalCharges": [((0, 1000), "abaixo de R$ 1.000")],
    "MonthlyIncome": [],                    # aqui o achado é a ausência de pico: linha da média por faixa
}
ORDEM_QUANT = ["tenure", "MonthlyCharges", "TotalCharges", "MonthlyIncome"]
BASE = "#cfd8dc"
invalidos = qualidade[qualidade["base"] == "historico"].set_index("coluna")["fora_do_dominio"]

fig, axes = plt.subplots(2, 2, figsize=(11, 7.8))
for ax, col in zip(axes.flat, ORDEM_QUANT):
    x = pd.to_numeric(hist_raw[col], errors="coerce").dropna()
    cont, bordas, barras = ax.hist(x, bins=BINS[col], color=BASE, edgecolor="white")
    topo = cont.max()
    ax.set_ylim(0, topo * 1.30)                                    # folga para rótulos e mediana
    for (ini, fim), rotulo in DESTAQUE_FAIXAS[col]:
        dentro = [i for i, b in enumerate(bordas[:-1]) if ini <= b < fim]
        for i in dentro:
            barras[i].set_facecolor(COR)
        n = int(cont[dentro].sum())
        primeira = dentro[0] == 0                                  # faixa no início do eixo: rótulo alinhado à esquerda
        xpos = bordas[0] if primeira else (bordas[dentro[0]] + bordas[dentro[-1] + 1]) / 2
        ax.text(xpos, cont[dentro].max() + topo * 0.03, f"{rotulo}\n{virg(n, 0)} ({virg(100 * n / len(x), 0)}%)",
                ha="left" if primeira else "center", va="bottom", fontsize=8.5, color=COR, fontweight="bold",
                bbox=dict(facecolor="white", edgecolor="none", pad=0.5))
    if not DESTAQUE_FAIXAS[col]:                                   # renda: todas as faixas perto da média
        media_faixa = cont.mean()
        ax.axhline(media_faixa, color=COR, linestyle=":", linewidth=1.5)
        ax.text(bordas[0], topo * 1.04,
                f"cada faixa tem entre {virg(cont.min(), 0)} e {virg(cont.max(), 0)} clientes",
                fontsize=8.5, color=COR, fontweight="bold", bbox=dict(facecolor="white", edgecolor="none", pad=0.5))
    med = x.median()
    ax.axvline(med, color=TEXTO, linestyle="--", linewidth=1)
    ax.text(med, topo * 1.28, f" mediana {virg(med, 0)}", fontsize=8.5, color=TEXTO, va="top",
            bbox=dict(facecolor="white", edgecolor="none", pad=1))
    n_inv = int(invalidos.get(col, 0))
    ax.set_title(col, pad=16)
    ax.text(0, 1.01, f"n = {virg(len(x), 0)}" + (f"  ·  {n_inv} inválidos fora do gráfico" if n_inv else ""),
            transform=ax.transAxes, fontsize=8, color="#666666", va="bottom")
    ax.set_ylabel("Clientes")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 0)))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 0)))
    ax.tick_params(length=0)

fig.suptitle("tenure e MonthlyCharges têm dois picos; TotalCharges se concentra em valores baixos;\n"
             "MonthlyIncome é quase uniforme. Em azul, as faixas que sustentam cada afirmação",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ORDEM_QUANT)
plt.show()

### Antes do boxplot: o critério de Tukey para valores extremos

**O que faço:** verifico se alguma variável quantitativa tem valores muito afastados dos demais (outliers).

**Por quê?** Um valor extremo pode ser erro de digitação (uma mensalidade de R$ 10.000), um cliente realmente atípico
ou um sinal de que a variável precisa de transformação. Na regressão logística, poucos pontos extremos podem puxar
os coeficientes. Preciso saber se eles existem **antes** de modelar.

> **Resumo: critério de Tukey (cercas do boxplot)**
>
> IQR = Q3 − Q1 (intervalo interquartil: onde estão os 50% centrais dos clientes)
>
> | Limite | Fórmula |
> |---|---|
> | Inferior | Q1 − 1,5 × IQR |
> | Superior | Q3 + 1,5 × IQR |
>
> Um valor fora desse intervalo é **candidato** a outlier.
>
> **Como ler o boxplot**
> - a caixa vai de Q1 a Q3, e a linha dentro dela é a mediana;
> - os "bigodes" vão até o último valor dentro dos limites;
> - os pontos isolados, além dos bigodes, são os outliers.
>
> **Por que Tukey, e não média ± 3 desvios-padrão?**
> - Os quartis não se deixam levar pelos próprios valores extremos; a média e o desvio-padrão, sim.
> - O critério não exige normalidade, e as distribuições aqui não são normais (seção 5.2).
> - Numa distribuição normal, os limites de Tukey ficam em cerca de ±2,7 desvios-padrão e deixam de fora só 0,7% dos dados.
>
> **Outlier não se apaga automaticamente.** Primeiro investigo: é erro, cliente atípico ou característica da variável?
> A decisão (corrigir, limitar ou manter) fica registrada na limpeza.

In [ ]:
# 5.4 Boxplot de cada variável quantitativa: há outliers?
#     Tracejado: limites de Tukey (Q1 - 1,5·IQR e Q3 + 1,5·IQR). Ponto fora deles = outlier, em laranja.
fig, axes = plt.subplots(4, 1, figsize=(10, 7.5))
outliers = {}
for ax, col in zip(axes, ORDEM_QUANT):
    x = pd.to_numeric(hist_raw[col], errors="coerce").dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers[col] = int(((x < lim_inf) | (x > lim_sup)).sum())

    ax.boxplot(x, vert=False, widths=0.5, patch_artist=True, showfliers=True,
               boxprops=dict(facecolor=COR, alpha=0.5, edgecolor=COR),
               medianprops=dict(color=TEXTO, linewidth=2),
               whiskerprops=dict(color=COR), capprops=dict(color=COR),
               flierprops=dict(marker="o", markersize=3, markerfacecolor=DESTAQUE, markeredgecolor="none"))
    for lim in (lim_inf, lim_sup):
        ax.axvline(lim, color="#999999", linestyle="--", linewidth=0.8)
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.set_title(col, pad=4, fontsize=10)
    ax.text(1, 1.02, f"{outliers[col]} fora dos limites de Tukey", transform=ax.transAxes,
            ha="right", fontsize=9, color=DESTAQUE if outliers[col] else "#666666")
    folga = (x.max() - x.min()) * 0.05
    ax.set_xlim(min(x.min(), lim_inf) - folga, max(x.max(), lim_sup) + folga)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 0)))

fig.suptitle("Nenhuma variável quantitativa tem outliers pelo critério de Tukey",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ORDEM_QUANT)
plt.show()
print("Outliers por variável:", outliers)

> **Como ler os gráficos 5.3 e 5.4**
>
> **5.3 (histogramas):** cada painel mostra quantos clientes há em cada faixa de valores.
> - **Barras em azul:** as faixas que sustentam o que o título afirma, com o número e o percentual de clientes escritos
>   ao lado. As outras ficam em cinza.
> - **Linha tracejada:** a mediana. Quando ela fica longe dos picos, a média e a mediana representam mal o "cliente típico".
> - **Linha pontilhada da renda:** em `MonthlyIncome`, o achado é a **ausência** de pico; a linha marca a média por faixa,
>   e todas as barras ficam perto dela.
> - **Texto sob o título:** quantos valores válidos entraram e quantos inválidos ficaram de fora (seção 4).
>
> **5.4 (boxplots):** cada linha é uma variável.
> - A caixa vai do 1º ao 3º quartil (os 50% centrais dos clientes), e a linha dentro dela é a mediana.
> - Os "bigodes" vão até os limites de Tukey (Q1 − 1,5 × IQR e Q3 + 1,5 × IQR), cortados no mínimo e no máximo observados.
> - **O que procurar:** pontos isolados além dos bigodes seriam outliers. Não há nenhum, em nenhuma variável.

> **Explicação**
>
> | Variável | CV | O que os números e gráficos mostram | Implicação |
> |---|---|---|---|
> | `tenure` | 76% | Dois picos: 1.089 clientes (19%) no primeiro semestre e 810 (14%) entre 66 e 72 meses. Curtose −1,39 confirma o achatamento | Há dois perfis de relacionamento; a média (32) não representa nenhum deles |
> | `MonthlyCharges` | 47% | Dois picos: 838 clientes (15%) entre R$ 20 e 30 (só telefonia) e 2.699 (48%) entre R$ 70 e 110 (pacotes com internet). Curtose −1,27 | O preço reflete o pacote contratado, não só o cliente |
> | `TotalCharges` | 99% | Única assimétrica (0,97): 2.205 clientes (41%) têm menos de R$ 1.000 acumulados, e poucos passam de R$ 6.000. Média 2.272 bem acima da mediana 1.391 | Usar a mediana como centro; a variável é quase `MonthlyCharges` × `tenure` |
> | `MonthlyIncome` | 56% | Praticamente uniforme de 100 a 10.000: cada faixa de 1.000 tem entre 478 e 600 clientes, sem a assimetria típica de renda real | Indício de variável gerada artificialmente; tende a não explicar o churn |
>
> **As quatro variáveis têm CV acima de 30%, mas por motivos diferentes:**
> - em `tenure` e `MonthlyCharges`, por causa dos dois perfis de cliente (novos e antigos; só telefonia e pacote com internet);
> - em `TotalCharges`, porque ela multiplica as duas anteriores e soma a dispersão de ambas;
> - em `MonthlyIncome`, porque os valores se espalham por igual de 100 a 10.000.
>
> **Outliers pelo critério de Tukey**
>
> | Variável | Q1 | Q3 | IQR | Limite inferior | Limite superior | Mínimo e máximo observados |
> |---|---|---|---|---|---|---|
> | `tenure` | 9 | 55 | 46 | −60 | 124 | 0 a 72 |
> | `MonthlyCharges` | 35,2 | 89,8 | 54,6 | −46,8 | 171,8 | 18,4 a 118,7 |
> | `TotalCharges` | 412,9 | 3.729,8 | 3.316,9 | −4.562,4 | 8.705,0 | 18,8 a 8.684,8 |
> | `MonthlyIncome` | 2.693,8 | 7.574,8 | 4.881,0 | −4.627,8 | 14.896,3 | 100 a 9.993 |
>
> - **Nenhum ponto fica fora dos limites, em nenhuma variável.** O caso mais próximo é o máximo de `TotalCharges`,
>   só 20 abaixo do limite superior.
> - Os limites inferiores são negativos porque as distribuições são largas. Como nenhuma dessas variáveis pode ser
>   negativa, só o limite superior importa aqui.
> - As distribuições são limitadas e sem caudas extremas: **não será preciso tratar outliers** na limpeza.
>
> **Para o negócio:** não há mensalidades ou tempos de casa absurdos. O desafio desta base não está em valores extremos,
> e sim nos dois perfis de cliente (novos e antigos; só telefonia e pacote com internet).

> **Anotações [André]: dúvidas para o grupo**
>
> 1. As faixas dos histogramas (6 meses, R$ 10, R$ 1.000) fazem sentido para o negócio, ou alguém sugere outros cortes?
> 2. `MonthlyIncome` é quase uniforme, o que não parece renda real. Alguém sabe se essa coluna foi gerada artificialmente pela organização?
>
> **Respostas e decisões do grupo:**
>
> - **2 (Carlos, 10/11):** sim, a renda veio de uma base externa, sem garantia de precisão, e pode ser ignorada (D12).

In [ ]:
# 5.6 Frequências das variáveis qualitativas (histórico), com resumo por variável
QUALI = meta.index[meta["tipo_estatistico"].str.startswith("qualitativa")].drop("Churn").tolist()

freq = []
for col in QUALI:
    s = hist_raw[col]
    for cat, n in s.value_counts().items():
        freq.append({"variavel": col, "categoria": cat, "n": int(n), "pct": round(100 * n / len(s), 2)})
freq = pd.DataFrame(freq)

resumo_quali = (freq.groupby("variavel", sort=False)
                .agg(n_categorias=("categoria", "size"),
                     categoria_mais_comum=("categoria", "first"),
                     pct_mais_comum=("pct", "max"),
                     n_categorias_raras=("pct", lambda p: int((p < 5).sum())))
                .reset_index())
resumo_quali["tipo_estatistico"] = resumo_quali["variavel"].map(meta["tipo_estatistico"])
display(resumo_quali)

print("Categorias raras (menos de 5% dos clientes):")
display(freq[freq["pct"] < 5])

In [ ]:
# 5.7 Barras das qualitativas: o olho vai direto para o que o título afirma
LINHAS_QUALI = {                                                   # painéis agrupados por tema, uma linha por tema
    "Perfil do cliente":     ["gender", "SeniorCitizen", "Partner", "Dependents"],
    "Contrato e conexão":    ["Contract", "PaymentMethod", "PaperlessBilling", "InternetService"],
    "Proteção e suporte":    ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"],
    "Telefonia e streaming": ["PhoneService", "MultipleLines", "StreamingTV", "StreamingMovies"],
}
PROTECAO = LINHAS_QUALI["Proteção e suporte"]
SEM_INTERNET = "#5f6b70"                                           # cinza-escuro: a mesma categoria repetida em 7 colunas
BASE = "#cfd8dc"                                                   # cinza-claro: o resto, que não é o assunto do título

def cor_barra(col, cat, pct):
    if pct < 5:
        return DESTAQUE                                            # categoria rara (problema de grafia)
    if (col == "Contract" and cat == "Month-to-month") or (col in PROTECAO and cat == "No"):
        return COR                                                 # o que o título afirma
    if cat == "No internet service" or (col == "InternetService" and cat == "No"):
        return SEM_INTERNET
    return BASE

NOTAS = {                                                          # outras observações, uma linha sob o título do painel
    "gender": "equilíbrio quase perfeito",
    "SeniorCitizen": "só 1 em cada 6 é idoso",
    "Contract": "mensal é mais que os outros dois somados",
    "PaymentMethod": "cheque eletrônico é o mais comum",
    "InternetService": "fibra é a mais comum",
    "OnlineSecurity": "metade não tem",
    "TechSupport": "metade não tem",
    "PhoneService": "quase todos têm; no e yes são erro de grafia",
    "StreamingTV": "Yes e No empatados",
    "StreamingMovies": "Yes e No empatados",
}

fig, axes = plt.subplots(4, 4, figsize=(15, 13))
for r, (tema, cols) in enumerate(LINHAS_QUALI.items()):
    for c, col in enumerate(cols):
        ax = axes[r, c]
        d = freq[freq["variavel"] == col].set_index("categoria")["pct"]
        ordem = [k for k in meta.loc[col, "dominio_esperado"] if k in d.index]   # mesma ordem em todos os painéis
        ordem += [k for k in d.index if k not in ordem]                          # inválidas no fim
        d = d.reindex(ordem)[::-1]                                               # primeira categoria no topo
        ax.barh(d.index, d.values, color=[cor_barra(col, k, v) for k, v in d.items()], height=0.62)
        for k, v in d.items():
            forte = cor_barra(col, k, v) in (COR, SEM_INTERNET, DESTAQUE)
            ax.text(v + 1.5, k, f"{virg(v, 0)}%", va="center", fontsize=9, fontweight="bold" if forte else "normal",
                    color=DESTAQUE if cor_barra(col, k, v) == DESTAQUE else TEXTO)
        ax.set_title(col, pad=16)                                                # mesma altura em todos os painéis
        if col in NOTAS:
            ax.text(0, 1.02, NOTAS[col], transform=ax.transAxes, fontsize=8.5, color="#666666", style="italic")
        ax.set_xlim(0, 105)                                                      # mesma escala em todos os painéis
        ax.set_xticks([])
        ax.spines[["bottom", "left"]].set_visible(False)
        ax.tick_params(axis="y", length=0, labelsize=9)
    axes[r, 0].annotate(tema.upper(), xy=(0, 1), xycoords="axes fraction", xytext=(-150, 34),
                        textcoords="offset points", fontsize=9, fontweight="bold", color="#888888")

from matplotlib.patches import Patch
legenda = [Patch(color=COR, label="o que o título afirma"),
           Patch(color=SEM_INTERNET, label="sem internet (a mesma categoria em 7 colunas)"),
           Patch(color=DESTAQUE, label="categoria rara, abaixo de 5% (grafia minúscula)"),
           Patch(color=BASE, label="demais categorias")]
fig.legend(handles=legenda, loc="upper left", bbox_to_anchor=(0.01, 0.962), ncol=4, frameon=False, fontsize=9)
fig.suptitle("A maioria tem contrato mensal (55%) e não tem os serviços de proteção (44% a 50%);\n"
             "22% não têm internet, e essa mesma categoria se repete em sete colunas",
             x=0.01, y=0.995, ha="left", fontsize=12, fontweight="bold")
rodape(fig, sum(LINHAS_QUALI.values(), []))
fig.subplots_adjust(top=0.885, hspace=0.55)
plt.show()

> **Explicação**
>
> **Como ler o gráfico**
> - **Cores:**
>   - azul: o que o título afirma;
>   - cinza-escuro: clientes sem internet;
>   - laranja: categorias raras;
>   - cinza-claro: o resto.
> - **Uma linha por tema:** perfil do cliente, contrato e conexão, proteção e suporte, telefonia e streaming.
> - **Ordem fixa das categorias** (a do domínio esperado, seção 3) em todos os painéis, e a mesma escala de 0 a 100%.
>   O olho compara o mesmo lugar de um painel para o outro. `Contract`, que é ordinal, fica na ordem natural (mensal → 1 ano → 2 anos).
>
> **O que o título afirma**
> - 55% estão em **contrato mensal**, sem fidelidade: mais que os contratos de 1 e 2 anos somados.
> - Os quatro **serviços de proteção e suporte** têm o "No" como maior categoria (44% a 50%). A linha azul inteira
>   mostra que isso vale para os quatro, não para um só.
> - 22% **não têm internet**, e essa mesma categoria aparece em `InternetService` e nos seis serviços.
>
> **Outras observações**
> - **Equilíbrio:** `gender` (50/50) e `Partner` (48/52) dividem a carteira ao meio.
> - **Desbalanceadas:** `SeniorCitizen` (84% não idosos) e `PhoneService` (89% com telefonia) têm uma categoria dominante.
>   Isso não é erro, mas variáveis assim tendem a separar pouco os clientes.
> - **Conexão:** a fibra ótica (44%) é a mais comum, à frente do DSL (34%).
> - **Pagamento:** o cheque eletrônico (34%) é a forma mais comum, e as duas automáticas somam 44%.
> - **Streaming:** Yes e No estão empatados (39% cada), diferente dos serviços de proteção.
> - **Categorias raras:** só o `yes` e o `no` minúsculos de `PhoneService` (0,5% e 0,7%), que são o erro de grafia da
>   seção 4 e somem na limpeza.
> - **Redundância:** "No internet service" (22%) e "No phone service" (10%) repetem a informação de `InternetService` e
>   `PhoneService`. A seção 6 mede isso.
>
> **Perguntas que levo para a seção 7 (associação com o churn)**
> - O contrato mensal, majoritário, é também o de maior churn?
> - Quem não tem proteção e suporte sai mais?
> - O cheque eletrônico, a forma mais comum de pagamento, está ligado ao churn?
> - O streaming, com Yes e No empatados, separa quem sai?

> **Anotações [André]: dúvidas para o grupo**
>
> 1. Faz sentido manter "No internet service" como categoria própria nos serviços, ou transformar em "No" e deixar a falta de internet só em `InternetService` (proposta da seção 6)?
> 2. Alguma característica que vocês esperavam ver na carteira e não aparece na base?
>
> **Respostas e decisões do grupo:**
>
> - **1 (D10, 10/11):** ◐ sugestão de transformar em sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; a seleção final é feita na modelagem.

In [ ]:
# 5.9 Histórico x inferência: os clientes a pontuar se parecem com os de treino?
#     Quantitativas: teste de Kolmogorov-Smirnov. Qualitativas: qui-quadrado de homogeneidade.
#     Todas: PSI (Population Stability Index), a medida de estabilidade usada em crédito.
def psi(esperado: pd.Series, observado: pd.Series) -> float:
    """PSI entre duas distribuições de proporções (mesmas categorias ou faixas)."""
    e, o = esperado.clip(lower=1e-4), observado.clip(lower=1e-4)
    return float(((o - e) * np.log(o / e)).sum())

linhas = []
for col in QUANT:
    h = pd.to_numeric(hist_raw[col], errors="coerce").dropna()
    i = pd.to_numeric(inf_raw[col], errors="coerce").dropna()
    ks, p = stats.ks_2samp(h, i)
    cortes = np.unique(h.quantile(np.linspace(0, 1, 11)))         # 10 faixas pelos decis do histórico
    ph = pd.cut(h, cortes, include_lowest=True).value_counts(normalize=True, sort=False)
    pi = pd.cut(i.clip(cortes[0], cortes[-1]), cortes, include_lowest=True).value_counts(normalize=True, sort=False)
    linhas.append({"variavel": col, "tipo": "quantitativa", "teste": "Kolmogorov-Smirnov",
                   "estatistica": ks, "p_valor": p, "PSI": psi(ph, pi),
                   "resumo": f"mediana {virg(h.median(), 1)} x {virg(i.median(), 1)}"})

for col in QUALI:
    corr = {"yes": "Yes", "no": "No"}                                       # grafia padronizada só para comparar
    h, i = hist_raw[col].replace(corr), inf_raw[col].replace(corr)
    tab = pd.crosstab(pd.concat([h, i]), ["hist"] * len(h) + ["inf"] * len(i))
    qui2, p, _, _ = stats.chi2_contingency(tab)
    ph, pi = tab["hist"] / tab["hist"].sum(), tab["inf"] / tab["inf"].sum()
    maior = (pi - ph).abs().idxmax()
    linhas.append({"variavel": col, "tipo": "qualitativa", "teste": "Qui-quadrado",
                   "estatistica": qui2, "p_valor": p, "PSI": psi(ph, pi),
                   "resumo": f"maior diferença: {maior} ({virg(100 * ph[maior], 1)}% x {virg(100 * pi[maior], 1)}%)"})

drift = pd.DataFrame(linhas)
drift["estabilidade"] = pd.cut(drift["PSI"], [-np.inf, 0.10, 0.25, np.inf],
                               labels=["estável", "atenção", "mudou"]).astype(str)
display(drift.sort_values("PSI", ascending=False).round(4))
print("Variáveis com p < 0,05:", drift.loc[drift["p_valor"] < 0.05, "variavel"].tolist() or "nenhuma")
print("PSI máximo:", virg(drift["PSI"].max(), 4))

### O que é o PSI e por que uso

**A pergunta:** os 1.409 clientes que o modelo vai pontuar se parecem com os 5.634 com que ele aprendeu?

**Uma analogia:** um professor monta uma prova com base na turma do ano passado. Se a turma deste ano for parecida, a
prova funciona. Se vier uma turma muito diferente (outra idade, outra formação), a mesma prova mede mal. O modelo é a
prova, o histórico é a turma do ano passado e a inferência é a turma deste ano. A mudança de perfil entre as duas
turmas se chama **drift**.

**O que o PSI faz:** compara, faixa por faixa (ou categoria por categoria), a **porcentagem** de clientes em cada base
e soma as diferenças num único número.

> **Como o PSI é calculado** (Population Stability Index)
>
> Para cada categoria ou faixa:
> - **A** = % na inferência − % no histórico → **quanto** mudou, em pontos;
> - **B** = ln(% na inferência ÷ % no histórico) → **quanto** mudou em proporção.
> - Na conta, as porcentagens entram como proporção: 55% vira 0,55. Por isso a diferença de 20 pontos aparece como 0,1999.
>
> PSI = soma de A × B em todas as categorias.
>
> **Por que multiplicar A por B?**
> - A e B têm sempre o mesmo sinal, então cada parcela é positiva. Aumentos e quedas não se anulam.
> - O B dá peso à mudança **relativa**: uma categoria que vai de 2% para 4% (dobrou) pesa mais que uma de 50% para 52%,
>   embora as duas tenham mudado 2 pontos.
>
> | PSI | Leitura (convenção do mercado de crédito) |
> |---|---|
> | Abaixo de 0,10 | Estável: a população é a mesma |
> | 0,10 a 0,25 | Atenção: houve mudança, investigar |
> | Acima de 0,25 | A população mudou: o modelo pode não valer mais |
>
> **Por que o PSI, e não só o p-valor?** O p-valor depende do tamanho da amostra. Com muitos clientes, qualquer
> diferença mínima fica "significativa"; com poucos, uma diferença grande pode passar. O PSI mede o **tamanho** da
> mudança, como o V de Cramér faz para a associação.

A 5.9a mostra a conta com `Contract` e com um cenário inventado em que o contrato mensal sobe de 55% para 75%.
A 5.9b coloca as 20 variáveis e esse cenário no mesmo gráfico.

In [ ]:
# 5.9a O PSI passo a passo: Contract real e um cenário inventado em que a carteira mudou
ph = hist_raw["Contract"].value_counts(normalize=True)
pi = inf_raw["Contract"].value_counts(normalize=True).reindex(ph.index)
CENARIO = pd.Series({"Month-to-month": 0.75, "One year": 0.13, "Two year": 0.12})   # inventado: mais contratos mensais

def tabela_psi(esperado, observado, nome_obs):
    t = pd.DataFrame({"% histórico": 100 * esperado, f"% {nome_obs}": 100 * observado})
    t["diferença (A)"] = (observado - esperado)
    t["ln(razão) (B)"] = np.log(observado / esperado)
    t["contribuição (A × B)"] = t["diferença (A)"] * t["ln(razão) (B)"]
    return t

real = tabela_psi(ph, pi, "inferência")
cenario = tabela_psi(ph, CENARIO.reindex(ph.index), "cenário")
PSI_CENARIO = cenario["contribuição (A × B)"].sum()
print(f"Contract real:     PSI = {virg(real['contribuição (A × B)'].sum(), 4)}")
display(real.round(4).reset_index(names="categoria"))
print(f"Cenário inventado: PSI = {virg(PSI_CENARIO, 4)}  (contrato mensal passando de 55% para 75%)")
display(cenario.round(4).reset_index(names="categoria"))

In [ ]:
# 5.9b Quanto é um PSI de 0,011? À esquerda, como seria uma mudança de verdade; à direita, o PSI das 20 variáveis
CINZA_HIST = "#9aa5aa"
fig, eixos = plt.subplot_mosaic([["real", "psi"], ["cen", "psi"]], figsize=(13, 8),
                                width_ratios=[1, 1.35])
ax_real, ax_cen, ax_psi = eixos["real"], eixos["cen"], eixos["psi"]

# esquerda: Contract no histórico contra a inferência real (em cima) e contra o cenário inventado (embaixo)
cats = ph.index[::-1]
for ax, outro, cor, nome, valor in [(ax_real, pi, COR, "inferência real", real["contribuição (A × B)"].sum()),
                                    (ax_cen, CENARIO.reindex(ph.index), DESTAQUE, "cenário inventado", PSI_CENARIO)]:
    y = np.arange(len(cats))
    ax.barh(y + 0.2, 100 * ph[cats], height=0.38, color=CINZA_HIST, label="histórico")
    ax.barh(y - 0.2, 100 * outro[cats], height=0.38, color=cor, label=nome)
    for yy, v1, v2 in zip(y, 100 * ph[cats], 100 * outro[cats]):
        ax.text(v1 + 1, yy + 0.2, f"{virg(v1, 0)}%", va="center", fontsize=8, color="#666666")
        ax.text(v2 + 1, yy - 0.2, f"{virg(v2, 0)}%", va="center", fontsize=8, color=cor, fontweight="bold")
    ax.set_yticks(y, cats)
    ax.set_xlim(0, 90)
    ax.set_xticks([])
    ax.spines[["bottom", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    ax.set_title(f"Contract: histórico x {nome}\nPSI = {virg(valor, 4 if valor < 0.01 else 2)}", fontsize=10,
                 color=cor, loc="left")

# direita: PSI de cada variável como ponto, sobre as faixas de leitura
d = drift.sort_values("PSI")
ax = ax_psi
ax.axvspan(0, 0.10, color="#eef3f5", zorder=0)
ax.axvspan(0.10, 0.25, color="#fbeee2", zorder=0)
ax.axvspan(0.25, 0.30, color="#f6dcc6", zorder=0)
for x0, rot in [(0.05, "estável"), (0.14, "atenção"), (0.275, "mudou")]:
    ax.text(x0, len(d) - 0.2, rot, ha="center", fontsize=9, color="#666666")
ax.hlines(d["variavel"], 0, d["PSI"], color=COR, linewidth=1.5, zorder=2)
ax.scatter(d["PSI"], d["variavel"], color=COR, s=30, zorder=3)
ax.axvline(PSI_CENARIO, color=DESTAQUE, linestyle="--", linewidth=1.5, zorder=2)
ax.text(PSI_CENARIO + 0.004, len(d) / 2, f"cenário inventado\n(painel à esquerda,\nembaixo)\nPSI = {virg(PSI_CENARIO, 2)}",
        fontsize=8.5, color=DESTAQUE, fontweight="bold", va="center")
ax.text(d["PSI"].max() + 0.006, d["variavel"].iloc[-1], f"maior: {virg(d['PSI'].max(), 3)}",
        fontsize=8.5, color=COR, fontweight="bold", va="center")
ax.set_xlim(0, 0.30)
ax.set_ylim(-0.7, len(d) + 0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 2)))
ax.set_xlabel("PSI (Population Stability Index)")
ax.tick_params(axis="y", length=0, labelsize=9)
ax.spines["left"].set_visible(False)
ax.set_title("PSI das 20 variáveis (histórico x inferência real)", fontsize=10, loc="left")

fig.suptitle("Histórico e inferência são praticamente iguais: nenhuma variável passa de PSI 0,011.\n"
             "Para comparar, a esquerda mostra como seria uma mudança de verdade (PSI 0,18)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, d["variavel"].tolist()[::-1], fonte="history.csv e inference.csv; cenário inventado para comparação")
plt.show()

> **Como ler o gráfico 5.9b**
>
> **Os dois painéis da esquerda são uma régua.** Eles mostram, com uma variável só (`Contract`), o que um número de PSI
> significa na prática:
> - **Em cima (azul), o caso real:** as barras do histórico (cinza) e da inferência (azul) têm praticamente o mesmo tamanho:
>   55% x 55%, 24% x 25%, 21% x 20%. Diferenças assim dão **PSI de 0,0006**.
> - **Embaixo (laranja), um cenário que eu inventei:** o contrato mensal sobe de 55% para 75%, e os outros dois caem
>   pela metade. A diferença salta aos olhos e dá **PSI de 0,18**. Esse cenário não existe nos dados; serve só para
>   mostrar como seria uma mudança de verdade.
>
> **O painel da direita é o resultado.** Cada ponto é o PSI de uma variável, comparando histórico e inferência real.
> - **As faixas de fundo** dão a leitura: azul-claro = estável (abaixo de 0,10), bege = atenção (0,10 a 0,25),
>   laranja-claro = a população mudou (acima de 0,25).
> - **A linha tracejada laranja** marca o 0,18 do cenário inventado, para comparar.
> - **O que procurar:** algum ponto perto ou à direita da linha laranja seria uma variável cujo perfil mudou entre
>   as bases. Não há nenhum: os 20 pontos estão colados no zero, e o maior (`tenure`, 0,011) fica longe até do fim
>   da faixa estável.
>
> **Em uma frase:** se a carteira tivesse mudado como no painel de baixo, veríamos pontos perto da linha laranja;
> como todos estão no zero, os clientes da inferência têm o mesmo perfil dos clientes do histórico.

> **Explicação (5.9): drift entre histórico e inferência**
>
> **Por que olhar isso:** o modelo aprende com o histórico e é aplicado na inferência. Se os clientes a pontuar
> forem diferentes dos de treino (outro perfil de contrato, de preço, de tempo de casa), o modelo erra mais do que
> as métricas de validação prometem. Essa mudança de distribuição se chama **drift**.
>
> **Como medi:**
> - **Quantitativas → Kolmogorov-Smirnov:** D = maior distância entre as duas distribuições acumuladas.
>   H0: as duas amostras vêm da mesma distribuição.
> - **Qualitativas → qui-quadrado de homogeneidade:** H0: as proporções de cada categoria são iguais nas duas bases.
> - **Todas → PSI (Population Stability Index):** Σ (% inferência − % histórico) × ln(% inferência ÷ % histórico),
>   por faixa ou categoria. Abaixo de 0,10, estável; de 0,10 a 0,25, atenção; acima de 0,25, a população mudou.
>
> **Resultado: as 20 variáveis estão estáveis.**
>
> | Medida | O que encontrei |
> |---|---|
> | p-valor | Nenhum abaixo de 0,05. O mais baixo é o de `PhoneService` (0,088) |
> | KS (quantitativas) | D de no máximo 0,023: as distribuições acumuladas nunca se afastam mais de 2,3 pontos percentuais |
> | Medianas | Praticamente iguais: `tenure` 29 x 28, `MonthlyCharges` 70,3 x 70,4, `TotalCharges` 1.390,7 x 1.412,4 |
> | Qualitativas | A maior diferença entre categorias é de 2,5 pontos percentuais (`StreamingTV` "No": 39,4% x 41,9%) |
> | PSI | Máximo de 0,011 (`tenure`), cerca de nove vezes abaixo do limite de atenção |
>
> **O cenário inventado (5.9a e 5.9b)**
> - **Para que serve:** o PSI é um número abstrato. Dizer que ele é 0,011 não ajuda quem nunca viu um PSI a saber se isso
>   é pouco ou muito, e as faixas de leitura (0,10 e 0,25) são uma convenção de mercado. O cenário funciona como uma
>   **régua**: mostra lado a lado como é uma mudança pequena e uma grande, e quanto PSI cada uma dá.
> - **Como montei:** peguei uma variável real, `Contract`, e inventei uma carteira em que o contrato mensal sobe de 55%
>   para 75%, enquanto o anual cai de 21% para 13% e o bienal de 24% para 12%. É uma mudança plausível no negócio, por
>   exemplo, depois de uma campanha agressiva de um concorrente ou do fim de uma política de fidelização. Calculei o PSI
>   dessa carteira inventada contra o histórico com a mesma fórmula usada nas variáveis reais.
> - **O que mostrou:**
>   - a carteira inventada dá **PSI de 0,18**, já na faixa de atenção, enquanto o `Contract` real dá 0,0006;
>   - no gráfico, as 20 variáveis reais ficam coladas no zero, longe da linha do cenário;
>   - a conta passo a passo (5.9a) mostra como o PSI funciona: a maior parcela não vem do mensal, que mudou mais em
>     pontos (+20), e sim do bienal, que **caiu pela metade**. O PSI pesa a mudança relativa, não só a absoluta.
> - **O que ele não é:** um dado da base. Uso só para explicar a escala; nenhuma conclusão sobre os clientes vem dele.
>
> **Conclusão: os clientes da inferência são do mesmo tipo dos clientes do histórico.** Na prática:
> 1. **Posso treinar com o histórico e aplicar direto na inferência.** Não é preciso reponderar os clientes, criar um
>    modelo separado nem ajustar o ponto de corte por mudança de perfil.
> 2. **As métricas da validação são uma boa estimativa do resultado na inferência.** É isso que dá confiança aos
>    números que vamos apresentar.
> 3. **A inferência parece ter sido separada ao acaso do mesmo conjunto de dados.** Diferenças de no máximo 2,5 pontos
>    são típicas de uma divisão aleatória, e não de clientes de outro período. Por isso, o churn esperado na inferência
>    é de cerca de 27%, como no histórico. Isso serve para conferir a previsão: se o modelo marcar muito mais ou muito
>    menos clientes como "Yes", algo está errado.
> 4. **O vazamento do texto também está na inferência.** 24,4% dos textos da inferência mencionam cancelamento, quase a
>    mesma proporção do histórico (25,2%). Se a avaliação do hackathon comparar a previsão com o churn real desses
>    clientes, um modelo com o texto vai pontuar muito alto, mesmo sem servir para o negócio. Isso pesa na decisão do
>    grupo e reforça a pergunta aos organizadores sobre como a entrega será avaliada.
>
> **O que este teste não garante:** ele compara só as variáveis explicativas (X). Mostra que **o perfil** dos clientes é
> o mesmo, mas não consegue verificar se **a relação** entre o perfil e o churn também é a mesma, porque a inferência não
> tem `Churn`. A primeira mudança se chama *drift de dados* (*covariate shift*); a segunda, *drift de conceito*. Num corte
> aleatório como este, o drift de conceito é improvável. Num uso real, ele só aparece depois, comparando a previsão com
> quem de fato cancelou (monitoramento, passo 7).
>
> **Para a apresentação:** treino e inferência têm o mesmo perfil (PSI máximo de 0,011), então o desempenho medido na
> validação deve se repetir nos clientes a pontuar.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. Alguém sabe como a organização separou histórico e inferência: sorteio ou por data? Isso confirma (ou não) a conclusão de que não há drift.
> 2. Vamos usar a taxa de cerca de 27% como checagem da previsão final antes de subir o arquivo?
>
> **Respostas e decisões do grupo:**
>
> - **2:** sim; a conferência da taxa prevista está no notebook 08, como passo final antes da entrega.
> - **1:** ainda sem resposta da organização.

> **Resumo da seção 5 e decisões**
>
> **O que encontrei:**
> - Nenhum outlier pelo critério de Tukey.
> - Distribuições com dois picos: clientes novos e antigos (`tenure`); só telefonia e pacote com internet (`MonthlyCharges`).
>   `TotalCharges` é assimétrica, e `MonthlyIncome` é quase uniforme.
> - Carteira com maioria em contrato mensal (55%), sem serviços de proteção (44% a 50%), e 22% sem internet.
> - Treino e inferência têm o mesmo perfil (PSI máximo de 0,011).
>
> **Decisões:**
>
> | Decisão | Por quê |
> |---|---|
> | Não tratar outliers | Não existem |
> | Usar testes não paramétricos nas seções 6 e 7 | As distribuições não são normais |
> | Usar a mediana como centro | A média representa mal distribuições com dois picos ou assimétricas |
> | Treinar com o histórico e aplicar direto na inferência | Não há drift |
> | Conferir a previsão final contra cerca de 27% de churn | Taxa esperada se a inferência for um corte aleatório da mesma base |

## 6. Associação entre as variáveis explicativas

**O que faço:** meço o quanto as variáveis explicativas se relacionam **entre si**, par a par, ainda sem olhar o churn.

**Por que antes do churn:** é a sequência da análise bivariada: primeiro cada variável sozinha
(seção 5), depois as variáveis entre si (esta seção) e só então a relação com o alvo (seção 7). Duas razões práticas:
- **Informação repetida.** Duas variáveis muito associadas contam a mesma história. No modelo, isso é a
  **multicolinearidade**: o efeito se divide entre as duas, os erros-padrão aumentam, e o sinal de um coeficiente pode
  até se inverter.
- **Entender o IV antes de vê-lo.** Se a mensalidade é quase o tipo de internet, o poder preditivo das duas vai
  parecer parecido na seção 7. Sabendo disso antes, leio o ranking do IV sem me enganar.

**A decisão fica para depois.** Quando duas variáveis se repetem, qual delas fica depende do poder preditivo (IV), que
só aparece na seção 7. Por isso, esta seção **descreve** as associações, e o veredito vem na seção 8.

> **Resumo: qual medida para cada par de tipos**
>
> | Par de tipos | Medida | Escala | Teoria (apêndice) | Pares |
> |---|---|---|---|---|
> | Qualitativa × qualitativa | V de Cramér | 0 a 1 | A.1 | 120 |
> | Quantitativa × quantitativa | Spearman (ρ), em módulo | 0 a 1 | A.2 | 6 |
> | Quantitativa × qualitativa | Razão de correlação (η) | 0 a 1 | A.3 | 64 |
>
> As três ficam na escala de 0 a 1 e podem ser lidas juntas, com uma ressalva: não são a mesma medida, então um 0,5 em ρ
> não é exatamente "igual" a um 0,5 em V. **Repetição forte:** a partir de 0,8, uma variável quase determina a outra.

> **Perguntas para guiar**
> - Alguma variável é quase uma cópia de outra?
> - As associações fortes vêm do comportamento dos clientes ou da forma como a base foi codificada?
> - Que grupos de variáveis andam juntos (cliente, contrato, serviços, financeiro)?

### Guia simples: o que é associação entre variáveis

#### Uma pergunta só

Todas as medidas desta seção respondem à mesma pergunta:

> **Se eu souber o valor de uma variável, consigo adivinhar melhor o valor da outra?**

- **0** = não ajuda nada: saber uma não muda o palpite sobre a outra.
- **1** = adivinho com certeza: uma determina a outra.

As três medidas (V de Cramér, Spearman e η) são só **ferramentas diferentes para a mesma pergunta**. A ferramenta depende do tipo das duas variáveis, como uma régua serve para medir comprimento e uma balança para medir peso.

#### A ferramenta para cada tipo de par, com exemplos dos nossos dados

| Par | Medida | Exemplo | O que o número quer dizer |
|---|---|---|---|
| Categoria × categoria | **V de Cramér** | `Partner` × `Dependents` = **0,45** | Com parceiro, 51% têm dependentes; sem parceiro, só 10%. Saber ajuda, mas não garante |
| | | `gender` × `Contract` ≈ **0** | Homens e mulheres têm os mesmos contratos (55% mensal nos dois). Saber o sexo não ajuda em nada |
| Número × número | **Spearman (ρ)** | `tenure` × `TotalCharges` = **0,89** | Até 12 meses, total mediano de R$ 173; de 13 a 36, R$ 1.406; de 37 a 72, R$ 4.254. **Sobem juntos** |
| | | `tenure` × `MonthlyCharges` = **0,28** | A mensalidade quase não muda com o tempo de casa (R$ 56, 70 e 80). Saber os meses ajuda pouco |
| Número × categoria | **Razão de correlação (η)** | `MonthlyCharges` × `InternetService` = **0,91** | Sem internet, cerca de R$ 21; DSL, R$ 58; fibra, R$ 92. Dentro de cada grupo os valores são parecidos: **sabendo o tipo de internet, quase acerto a conta** |

#### Como ler o número

| Valor | Leitura | O que fazemos |
|---|---|---|
| abaixo de 0,3 | Fraca | Nada: cada variável traz a sua informação |
| 0,3 a 0,5 | Moderada | Anotar |
| 0,5 a 0,8 | Moderada a forte | Olhar com atenção na seleção de variáveis |
| **0,8 ou mais** | **Repetição** | As duas contam a mesma história: **uma sobra** |

#### Cuidado: associação aparente

O exemplo clássico: **vendas de sorvete e afogamentos sobem juntos**. Um não causa o outro: os dois aumentam no **verão**. É um terceiro fator que cria a associação.

Nos nossos dados, o "verão" é **não ter internet**:
- `OnlineSecurity` × `TechSupport` dá **V = 0,73**, quase uma repetição.
- Mas olhando a tabela, a associação vem de uma única célula: os **1.233 clientes sem internet** aparecem como "No internet service" **nas duas colunas ao mesmo tempo**.
- **Tirando esses clientes, o V cai para 0,27**: quem tem segurança online contrata suporte em 54% dos casos; quem não tem, em 27%. Existe uma relação, mas fraca.
- O mesmo vale para todos os serviços: de 0,71 a 0,77 com todos os clientes, de 0,04 a 0,43 só com quem tem internet.

**Por isso a D10:** com os serviços como sim/não, e a falta de internet só em `InternetService`, a falsa repetição desaparece.

#### Por que isso importa para o modelo

- **Dois termômetros na mesma sala:** o segundo não traz informação nova. Uma variável que repete outra não melhora o modelo.
- **Dois colegas empurrando o mesmo carro:** dá para ver que o carro andou, mas não quanto cada um empurrou. Na regressão logística, duas variáveis repetidas dividem o efeito entre si, e os coeficientes ficam instáveis, às vezes até com o sinal trocado. Isso se chama **multicolinearidade**.
- **Associação não é causa:** `MonthlyCharges` e `InternetService` estão ligadas porque o pacote define o preço, e não o contrário. A associação diz que andam juntas, não por quê.
- **Ainda sem o churn:** esta seção olha as variáveis explicativas **entre si**. A relação de cada uma com o churn é a seção 7.

#### Em três frases, para a banca

1. Medimos o quanto saber uma variável ajuda a adivinhar a outra, numa escala de 0 a 1.
2. Encontramos repetições reais (`tenure` × `TotalCharges`, mensalidade × pacote, `PhoneService` × `MultipleLines`) e uma repetição aparente nos serviços, criada pelos clientes sem internet.
3. Por isso transformamos os serviços em sim/não (D10) e vamos escolher entre as variáveis repetidas na seleção (D15 e D16).


In [ ]:
# 6.1 Pares de variáveis explicativas do mesmo tipo com associação de 0,5 ou mais, com a classificação de cada par
import itertools
CORR_GRAFIA = {"yes": "Yes", "no": "No"}          # grafia de PhoneService padronizada só para a análise

def v_cramer(a: pd.Series, b: pd.Series) -> float:
    """V de Cramér entre duas qualitativas: 0 = independentes, 1 = uma determina a outra."""
    t = pd.crosstab(a, b)
    qui2 = stats.chi2_contingency(t)[0]
    return float(np.sqrt(qui2 / (len(a) * (min(t.shape) - 1))))

SERVICOS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
num = hist_raw[QUANT].apply(pd.to_numeric, errors="coerce")
quali = {c: hist_raw[c].replace(CORR_GRAFIA) for c in QUALI}
com_internet = hist_raw["InternetService"].ne("No")

pares = []
for a, b in itertools.combinations(QUANT, 2):
    rho = num[[a, b]].dropna().corr(method="spearman").iloc[0, 1]
    pares.append({"variavel_1": a, "variavel_2": b, "medida": "Spearman", "associacao": rho, "so_quem_tem_internet": np.nan})
for a, b in itertools.combinations(QUALI, 2):
    so_int = (v_cramer(quali[a][com_internet], quali[b][com_internet])
              if a in SERVICOS and b in SERVICOS else np.nan)          # tira a categoria "No internet service"
    pares.append({"variavel_1": a, "variavel_2": b, "medida": "V de Cramér",
                  "associacao": v_cramer(quali[a], quali[b]), "so_quem_tem_internet": so_int})

pares = pd.DataFrame(pares)
pares["abs"] = pares["associacao"].abs()
redund = pares[pares["abs"] >= 0.5].sort_values("abs", ascending=False).drop(columns="abs").reset_index(drop=True)

def classificar_par(r) -> str:
    """Repetição real (≥ 0,8), aparente (vem só de 'No internet service') ou associação moderada."""
    if abs(r["associacao"]) >= 0.8:
        return "Repetição real"
    par = {r["variavel_1"], r["variavel_2"]}
    internet_x_servico = "InternetService" in par and bool(par & set(SERVICOS))
    cai_sem_internet = pd.notna(r["so_quem_tem_internet"]) and r["so_quem_tem_internet"] < 0.5
    if internet_x_servico or cai_sem_internet:
        return "Repetição aparente (No internet service)"
    return "Associação moderada"

redund["classificacao"] = redund.apply(classificar_par, axis=1)
print(redund["classificacao"].value_counts().to_string(), "\n")
print(f"{len(redund)} pares com associação de 0,5 ou mais (de {len(pares)} pares testados)")
display(redund.round(3))

> **Como ler a tabela 6.1**
>
> Cada linha é um **par de variáveis explicativas** do mesmo tipo (qualitativa × qualitativa ou quantitativa ×
> quantitativa). Testei os 126 pares possíveis desses tipos e mostro só os 24 com associação de 0,5 ou mais, do mais forte
> para o mais fraco. Os 64 pares mistos entram na matriz da 6.2.
>
> | Coluna | O que é |
> |---|---|
> | `variavel_1` e `variavel_2` | O par comparado |
> | `medida` | Spearman para duas quantitativas (A.2); V de Cramér para duas qualitativas (A.1) |
> | `associacao` | A força da relação entre as duas, de 0 a 1. Acima de 0,8, uma quase determina a outra |
> | `so_quem_tem_internet` | O mesmo V, recalculado sem os 1.233 clientes sem internet. Só existe para pares entre dois serviços; nos outros pares fica vazio (`null`), porque a categoria "No internet service" não existe ali ou é a própria variável que a define (`InternetService`) |
> | `classificacao` | O tipo do par: **repetição real** (≥ 0,8), **repetição aparente** (vem só de "No internet service") ou **associação moderada** (entre 0,5 e 0,8, de outra origem) |
>
> **Três grupos de pares na tabela:**
>
> | Grupo | Pares | Leitura |
> |---|---|---|
> | Repetição real | `PhoneService` × `MultipleLines` (1,00) e `tenure` × `TotalCharges` (0,89) | Uma variável quase determina a outra |
> | Repetição aparente | Os 15 pares entre serviços e os 6 de `InternetService` com cada serviço (0,71 a 0,77) | Vem da categoria "No internet service", igual em todas as colunas. Sem ela, a associação cai para 0,04 a 0,43 (6.3) |
> | Associação moderada | `MonthlyCharges` × `TotalCharges` (0,64) | Relação esperada (o total soma as mensalidades), mas abaixo de 0,8 |
>
> **O que procurar:** a diferença entre `associacao` e `so_quem_tem_internet`. Quando ela é grande, a associação não é
> do serviço em si, e sim da forma como a base foi codificada.

In [ ]:
# 6.2 Matriz de associação entre todas as variáveis explicativas: a medida certa para cada par de tipos
def eta(x: pd.Series, grupos: pd.Series) -> float:
    """Razão de correlação (eta): quanto da variação de x é explicada pelos grupos. 0 = nada, 1 = tudo."""
    ok = x.notna()
    x, grupos = x[ok], grupos[ok]
    entre = sum(len(g) * (g.mean() - x.mean()) ** 2 for _, g in x.groupby(grupos))
    total = ((x - x.mean()) ** 2).sum()
    return float(np.sqrt(entre / total))

ORDEM_MATRIZ = (["gender", "SeniorCitizen", "Partner", "Dependents", "MonthlyIncome"] +          # cliente
                ["tenure", "Contract", "PaperlessBilling", "PaymentMethod"] +                     # contrato
                ["PhoneService", "MultipleLines", "InternetService"] + SERVICOS +                 # serviços
                ["MonthlyCharges", "TotalCharges"])                                              # financeiro
eh_quant = {c: c in QUANT for c in ORDEM_MATRIZ}

n = len(ORDEM_MATRIZ)
valor = pd.DataFrame(np.nan, index=ORDEM_MATRIZ, columns=ORDEM_MATRIZ)
medida = pd.DataFrame("", index=ORDEM_MATRIZ, columns=ORDEM_MATRIZ)
for i, a in enumerate(ORDEM_MATRIZ):
    for b in ORDEM_MATRIZ[:i]:
        if eh_quant[a] and eh_quant[b]:
            v, m = abs(num[[a, b]].dropna().corr(method="spearman").iloc[0, 1]), "ρ"
        elif not eh_quant[a] and not eh_quant[b]:
            v, m = v_cramer(quali[a], quali[b]), "V"
        else:
            q, c = (a, b) if eh_quant[a] else (b, a)
            v, m = eta(num[q], quali[c]), "η"
        valor.loc[a, b], medida.loc[a, b] = v, m

pares_todos = (valor.stack().dropna().rename("associacao").reset_index()
               .rename(columns={"level_0": "variavel_1", "level_1": "variavel_2"}))
pares_todos["medida"] = [medida.loc[a, b] for a, b in zip(pares_todos["variavel_1"], pares_todos["variavel_2"])]
print(f"{len(pares_todos)} pares: " + ", ".join(f"{k} = {v}" for k, v in pares_todos["medida"].value_counts().items()))
print("Pares mistos (η) com 0,5 ou mais:")
display(pares_todos[(pares_todos["medida"] == "η") & (pares_todos["associacao"] >= 0.5)]
        .sort_values("associacao", ascending=False).round(3))

# gráfico: só o triângulo de baixo (a matriz é simétrica), cor = força da associação
from matplotlib.colors import LinearSegmentedColormap
mapa = LinearSegmentedColormap.from_list("assoc", ["#ffffff", "#cfe0e7", COR, "#0d3a4a"])
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(valor.values, cmap=mapa, vmin=0, vmax=1)
for i in range(n):
    for j in range(i):
        v = valor.iat[i, j]
        if v >= 0.10:                                              # abaixo de 0,10, deixo a célula sem número
            ax.text(j, i, f"{virg(v, 2)}\n{medida.iat[i, j]}", ha="center", va="center", fontsize=6.5,
                    color="white" if v > 0.55 else TEXTO)
        if v >= 0.8:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=DESTAQUE, linewidth=2.2))
for corte in [5, 9, 18]:                                           # separa os grupos
    ax.axhline(corte - 0.5, color="#888888", linewidth=0.8)
    ax.axvline(corte - 0.5, color="#888888", linewidth=0.8)
from matplotlib.transforms import blended_transform_factory
lateral = blended_transform_factory(ax.transAxes, ax.transData)    # x em fração do eixo, y nas linhas da matriz
for ini, fim, nome in [(0, 5, "CLIENTE"), (5, 9, "CONTRATO"), (9, 18, "SERVIÇOS"), (18, 20, "FINANC.")]:
    ax.text(-0.25, (ini + fim - 1) / 2, nome, transform=lateral, rotation=90, ha="center", va="center",
            fontsize=8.5, color="#777777", fontweight="bold")      # nome do grupo à esquerda, ao lado das linhas
ax.set_xticks(range(n), ORDEM_MATRIZ, rotation=60, ha="right", fontsize=8.5)
ax.set_yticks(range(n), ORDEM_MATRIZ, fontsize=8.5)
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
barra = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
barra.set_label("associação (0 = nenhuma, 1 = uma determina a outra)")
barra.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
fortes = pares_todos[pares_todos["associacao"] >= 0.8]
fig.suptitle(f"{len(fortes)} pares passam de 0,8 (contorno laranja): a mensalidade é quase determinada pelos serviços\n"
             "(η de 0,91 com InternetService), e o bloco escuro dos serviços vem da categoria 'No internet service'",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ORDEM_MATRIZ)
plt.show()

> **Como ler a matriz 6.2**
> - **Cada célula** é um par de variáveis: o número é a força da associação (0 a 1), e a letra embaixo é a medida usada
>   (ρ, V ou η). Só aparece o triângulo de baixo, porque a matriz é simétrica (A × B = B × A).
> - **Cor:** quanto mais escuro, mais forte. Células abaixo de 0,10 ficam sem número, para não poluir.
> - **Contorno laranja:** associação de 0,8 ou mais, a repetição forte que pesa no veredito da seção 8.
> - **Linhas cinza:** separam os grupos (cliente, contrato, serviços, financeiro), com o nome à esquerda.
>
> **O que a matriz mostra**
> 1. **Os dados pessoais quase não se relacionam com nada.** O canto de cima é quase todo branco. A exceção é
>    `Partner` × `Dependents` (0,45), o exemplo da A.1.
> 2. **O bloco escuro dos serviços** (V em torno de 0,72) vem da categoria "No internet service", como mostram a 6.3
>    e a 6.3b. Não é repetição real.
> 3. **Cinco pares passam de 0,8:**
>    - `PhoneService` × `MultipleLines` (V = 1): "No phone service" coincide com quem não tem telefonia;
>    - `tenure` × `TotalCharges` (ρ = 0,89): o total acumulado cresce com o tempo de casa;
>    - **`MonthlyCharges` × `InternetService` (η = 0,91)**, × `StreamingTV` (0,84) e × `StreamingMovies` (0,83): a
>      mensalidade é, em boa parte, a soma dos serviços contratados.
> 4. **`Contract` × `tenure`** (η = 0,67): contratos longos têm clientes mais antigos. É uma associação forte, mas abaixo
>    de 0,8.
>
> **O achado que só a matriz mostra:** a relação entre a mensalidade e os serviços não aparecia na 6.1, porque os pares
> mistos não estavam lá. A parte útil da mensalidade é o que ela tem **além** dos serviços: um cliente que paga mais do
> que a média do seu tipo de internet.

In [ ]:
# 6.2a V de Cramér: as 16 qualitativas entre si
valor_sim = valor.combine_first(valor.T)                           # matriz simétrica, a partir da 6.2
QUALI_M = [c for c in ORDEM_MATRIZ if not eh_quant[c]]
QUANT_M = [c for c in ORDEM_MATRIZ if eh_quant[c]]
mapa_assoc = LinearSegmentedColormap.from_list("assoc", ["#ffffff", "#cfe0e7", COR, "#0d3a4a"])

def desenhar(ax, m, triangulo):
    """Desenha a matriz m (0 a 1) com números, contorno laranja em ≥ 0,8 e, se pedido, só o triângulo de baixo."""
    vals = m.values.astype(float).copy()
    if triangulo:
        vals[np.triu_indices_from(vals)] = np.nan
    im = ax.imshow(vals, cmap=mapa_assoc, vmin=0, vmax=1)
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            if np.isnan(v):
                continue
            if v >= 0.10:
                ax.text(j, i, virg(v, 2), ha="center", va="center", fontsize=7.5, color="white" if v > 0.55 else TEXTO)
            if v >= 0.8:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=DESTAQUE, linewidth=2.2))
    ax.set_xticks(range(m.shape[1]), m.columns, rotation=60, ha="right", fontsize=8.5)
    ax.set_yticks(range(m.shape[0]), m.index, fontsize=8.5)
    ax.tick_params(length=0)
    for lado in ax.spines.values():
        lado.set_visible(False)
    return im

fig, ax = plt.subplots(figsize=(11, 9.5))
im = desenhar(ax, valor_sim.loc[QUALI_M, QUALI_M], triangulo=True)
barra = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
barra.set_label("V de Cramér (0 = independentes, 1 = uma determina a outra)")
barra.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
fig.suptitle("Entre as qualitativas, só PhoneService × MultipleLines é repetição real (V = 1);\n"
             "o bloco escuro dos serviços (V ≈ 0,72) vem da categoria 'No internet service' (6.3)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, QUALI_M)
plt.show()

In [ ]:
# 6.2b As quantitativas: Spearman entre si (esquerda) e razão de correlação η com cada qualitativa (direita)
fig, eixos = plt.subplot_mosaic([["rho", "eta"]], figsize=(15, 5.3), width_ratios=[4, 16])
ax1, ax2 = eixos["rho"], eixos["eta"]
desenhar(ax1, valor_sim.loc[QUANT_M, QUANT_M], triangulo=True)
ax1.set_title("Spearman (ρ), em módulo", loc="left", fontsize=10.5)
im = desenhar(ax2, valor_sim.loc[QUANT_M, QUALI_M], triangulo=False)
ax2.set_yticklabels([])
ax2.set_title("Razão de correlação (η): quanto cada qualitativa explica da quantitativa", loc="left", fontsize=10.5)
barra = fig.colorbar(im, ax=ax2, fraction=0.012, pad=0.01)
barra.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
fig.suptitle("tenure × TotalCharges (ρ = 0,89) e a mensalidade com o tipo de internet (η = 0,91) e os streamings são\n"
             "as repetições fortes; MonthlyIncome não se relaciona com nada",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, QUANT_M + ["InternetService", "StreamingTV", "StreamingMovies", "Contract"])
plt.show()

> **Como ler as matrizes 6.2a e 6.2b**
>
> São os mesmos números da matriz 6.2, separados por teste. Cada célula mostra a força da associação (0 a 1), e o
> contorno laranja marca a repetição forte (≥ 0,8).
>
> **6.2a, V de Cramér (A.1), as 16 qualitativas entre si:**
> - O canto de cima, quase branco, são os dados pessoais: pouco se relacionam com o resto. A exceção é
>   `Partner` × `Dependents` (0,45), o exemplo da A.1.
> - O único contorno laranja é `PhoneService` × `MultipleLines` (V = 1): "No phone service" coincide com quem não tem telefonia.
> - O bloco escuro embaixo, à direita, são os serviços (V ≈ 0,72). A 6.3 mostra que ele vem da categoria
>   "No internet service", e não de os clientes contratarem os serviços juntos.
>
> **6.2b, as 4 quantitativas:**
> - **À esquerda, Spearman (A.2) entre as quantitativas:** `tenure` × `TotalCharges` (0,89) é repetição forte;
>   `MonthlyCharges` × `TotalCharges` (0,64) é forte, mas abaixo de 0,8.
> - **À direita, η (A.3): quanto cada qualitativa explica de cada quantitativa.** Cada linha é uma quantitativa.
>   - `MonthlyCharges` tem a linha mais escura: a mensalidade é quase determinada pelo tipo de internet (0,91) e pelos
>     serviços contratados (0,77 a 0,84);
>   - `tenure` se destaca só com `Contract` (0,67): contratos longos têm clientes mais antigos;
>   - `MonthlyIncome` tem a linha toda branca: a renda não se relaciona com nenhuma outra variável.
> - **O que procurar:** linhas muito escuras indicam uma quantitativa que é, em boa parte, consequência das qualitativas.
>   É o caso da mensalidade, e por isso a seção 8 a coloca em "Atenção".

In [ ]:
# 6.3 Antes e depois: os serviços só parecem repetidos por causa da categoria "No internet service"
s = redund.dropna(subset=["so_quem_tem_internet"]).copy()
s["par"] = s["variavel_1"] + " × " + s["variavel_2"]
s = s.sort_values("so_quem_tem_internet").reset_index(drop=True)
LIMITE_FORTE = 0.8

fig, ax = plt.subplots(figsize=(10, 7))
for i, r in s.iterrows():
    ax.annotate("", xy=(r["so_quem_tem_internet"] + 0.012, i), xytext=(r["associacao"] - 0.012, i),
                arrowprops=dict(arrowstyle="-|>", color=CINZA, linewidth=1.6, mutation_scale=12))
    ax.text(r["associacao"] + 0.02, i, virg(r["associacao"], 2), va="center", ha="left", fontsize=8, color="#777777")
    ax.text(r["so_quem_tem_internet"] - 0.02, i, virg(r["so_quem_tem_internet"], 2), va="center", ha="right",
            fontsize=8, color=COR, fontweight="bold")
ax.scatter(s["associacao"], s.index, color=CINZA, s=50, zorder=3)
ax.scatter(s["so_quem_tem_internet"], s.index, color=COR, s=50, zorder=3)

topo = len(s) - 1                                                  # rótulos diretos na primeira linha, sem legenda
ax.text(s.loc[topo, "associacao"], topo + 0.7, "antes:\ntodos os clientes", ha="center", va="bottom",
        fontsize=9, color="#777777")
ax.text(s.loc[topo, "so_quem_tem_internet"], topo + 0.7, "depois:\nsó quem tem internet", ha="center",
        va="bottom", fontsize=9, color=COR, fontweight="bold")

ax.axvline(LIMITE_FORTE, color="#999999", linestyle=":", linewidth=1)
ax.text(LIMITE_FORTE + 0.01, -0.9, f"repetição forte ({virg(LIMITE_FORTE, 1)})", fontsize=8, color="#666666", va="bottom")
ax.set_yticks(s.index, s["par"])
ax.set_ylim(-1, len(s) + 1.2)
ax.set_xlim(-0.08, 1)
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
ax.set_xlabel("V de Cramér (0 = independentes, 1 = uma determina a outra)")
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
fig.suptitle(f"Antes, os serviços pareciam repetidos (V de {virg(s['associacao'].min(), 2)} a {virg(s['associacao'].max(), 2)});\n"
             f"depois de tirar quem não tem internet, a associação cai para {virg(s['so_quem_tem_internet'].min(), 2)} a "
             f"{virg(s['so_quem_tem_internet'].max(), 2)}\nNenhum par chega perto da repetição forte",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, SERVICOS + ["InternetService"])
plt.show()

In [ ]:
# 6.3b Por que a associação cai: a tabela cruzada de um par, antes e depois
A, B = "OnlineSecurity", "TechSupport"
ORD = ["No", "Yes", "No internet service"]
antes = pd.crosstab(hist_raw[A], hist_raw[B]).reindex(index=ORD, columns=ORD)
depois = pd.crosstab(hist_raw.loc[com_internet, A], hist_raw.loc[com_internet, B]).reindex(index=ORD[:2], columns=ORD[:2])
n_sem = int((~com_internet).sum())

from matplotlib.colors import LinearSegmentedColormap
mapa = LinearSegmentedColormap.from_list("azul", ["#ffffff", COR])
fig, axes = plt.subplots(1, 2, figsize=(11, 5.4))
casos = [(axes[0], antes, "Antes: todos os clientes", v_cramer(hist_raw[A], hist_raw[B])),
         (axes[1], depois, "Depois: só quem tem internet",
          v_cramer(hist_raw.loc[com_internet, A], hist_raw.loc[com_internet, B]))]
for ax, tab, nome, v in casos:
    pct = tab / tab.values.sum()
    ax.imshow(pct.values, cmap=mapa, vmin=0, vmax=0.45)
    for i in range(tab.shape[0]):
        for j in range(tab.shape[1]):
            cor_txt = "white" if pct.values[i, j] > 0.25 else TEXTO
            ax.text(j, i, f"{virg(tab.values[i, j], 0)}\n{virg(100 * pct.values[i, j], 1)}%",
                    ha="center", va="center", fontsize=9, color=cor_txt)
    rot = [c.replace("No internet service", "No internet\nservice") for c in tab.columns]
    ax.set_xticks(range(tab.shape[1]), rot)
    ax.set_yticks(range(tab.shape[0]), [c.replace("No internet service", "No internet\nservice") for c in tab.index])
    ax.set_xlabel(B)
    ax.set_ylabel(A)
    ax.tick_params(length=0)
    for lado in ["left", "bottom"]:
        ax.spines[lado].set_visible(False)
    ax.set_title(f"{nome} (V = {virg(v, 2)})", fontsize=11)
axes[1].set_xlim(-0.5, 2.5)                                       # mesma grade do "antes": o espaço vazio
axes[1].set_ylim(2.5, -0.5)                                       # é a categoria que foi retirada
axes[1].text(2, 2, f"{virg(n_sem, 0)} clientes\nsem internet\nretirados", ha="center", va="center",
             fontsize=9, color="#999999", style="italic")
axes[0].add_patch(plt.Rectangle((1.5, 1.5), 1, 1, fill=False, edgecolor=DESTAQUE, linewidth=2.5))
fig.suptitle(f"A associação alta vinha de uma única célula: os {virg(n_sem, 0)} clientes sem internet\n"
             f"aparecem como 'No internet service' nas duas colunas ao mesmo tempo",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, [A, B, "InternetService"])
plt.show()

> **Como ler os gráficos 6.3 e 6.3b**
>
> **6.3 (setas):** cada linha é um par de serviços.
> - O **ponto cinza** é o V de Cramér com todos os clientes (antes); o **ponto azul**, só com quem tem internet (depois).
> - A seta mostra quanto a associação cai quando tiro a categoria "No internet service".
> - A **linha pontilhada em 0,8** marca a repetição forte. Antes, todos os pares ficam perto de 0,7; depois, nenhum
>   passa de 0,43.
> - **O que procurar:** setas longas indicam associação que vinha da codificação da base, e não do serviço em si. É o
>   caso de todos os 15 pares.
>
> **6.3b (tabelas cruzadas):** mostra o motivo com um par, `OnlineSecurity` × `TechSupport`.
> - Cada célula traz o número de clientes e o percentual do total. Quanto mais escuro o azul, mais clientes.
> - **Antes**, a célula em laranja concentra os 1.233 clientes sem internet (21,9%), que aparecem como "No internet
>   service" nas duas colunas ao mesmo tempo. Essa coincidência obrigatória é o que faz o V parecer alto.
> - **Depois**, essa linha e essa coluna somem (o espaço vazio é a categoria retirada). Nas quatro células que sobram,
>   saber se o cliente tem segurança online diz pouco sobre se ele tem suporte técnico (V = 0,27).

> **Explicação**
>
> **1. Repetição real em cinco pares** (associação ≥ 0,8, contorno laranja na 6.2):
>
> | Par | Medida | Por quê |
> |---|---|---|
> | `PhoneService` × `MultipleLines` | V = 1,00 | "No phone service" aparece exatamente para quem não tem telefonia |
> | `tenure` × `TotalCharges` | ρ = 0,89 | O total cobrado é, em boa parte, mensalidade × tempo de casa |
> | `MonthlyCharges` × `InternetService` | η = 0,91 | A mensalidade é quase determinada pelo tipo de internet |
> | `MonthlyCharges` × `StreamingTV` e × `StreamingMovies` | η = 0,84 e 0,83 | Cada serviço contratado soma à mensalidade |
>
> **2. Os serviços parecem repetidos, mas não são (6.3 e 6.3b).** Com todos os clientes, os pares de serviços têm V entre
> 0,71 e 0,77. Quando tiro quem não tem internet, a associação cai para valores entre 0,04 e 0,43. A "repetição" vinha só da
> categoria "No internet service", que é a mesma nas seis colunas e em `InternetService`. Na hora de codificar, basta
> transformar cada serviço em "tem" (1) ou "não tem" (0); a falta de internet fica registrada só em `InternetService`.
>
> **3. Associações fortes, mas abaixo de 0,8:** `Contract` × `tenure` (η = 0,67) e `MonthlyCharges` × `TotalCharges`
> (ρ = 0,64). As duas fazem sentido no negócio e ficam para a seleção de variáveis decidir.
>
> **4. Os dados pessoais são quase independentes de todo o resto.** Gênero, idade e renda praticamente não se relacionam
> com contrato, serviços ou preço. Só `Partner` × `Dependents` (0,45) se destaca.
>
> **O que levo para as próximas seções:**
> - na **seção 7**, se o streaming e a mensalidade tiverem IV alto, já sei que parte disso vem da internet;
> - na **seção 8**, quando duas variáveis se repetem (≥ 0,8), fica a de maior IV, e a outra recebe "Atenção".
>
> **Para o negócio:** o preço que o cliente paga é, quase todo, consequência do pacote que ele contratou. Para entender o
> churn pelo preço, o que interessa é se ele paga **mais ou menos do que clientes com o mesmo pacote**.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. Concordam em codificar cada serviço como 1/0 e deixar a falta de internet só em `InternetService`?
> 2. Para separar o preço do pacote, proponho a mensalidade relativa (preço ÷ média do mesmo tipo de internet). Alguém tem outra ideia?
> 3. `Contract` × `tenure` (0,67): mantemos as duas e deixamos a seleção decidir?
>
> **Respostas e decisões do grupo:**
>
> - **1 (D10):** ◐ sugestão de sim/não, com as duas versões na ABT.
> - **2 (D16):** a `mensalidade_relativa_pacote` foi criada no 04 (média do pacote aprendida só no treino).
> - **3:** sim; as duas estão nas recomendadas, e a repetição entre elas é aceitável no modelo (VIF máximo de 2,75 na seção 5.4 do 05).

> **Resumo da seção 6 e decisões**
>
> **O que encontrei:** cinco pares com repetição forte (≥ 0,8): `PhoneService` × `MultipleLines`, `tenure` ×
> `TotalCharges` e a mensalidade com o tipo de internet e os dois streamings. A associação entre os serviços (V em torno
> de 0,72) era só a categoria "No internet service". Os dados pessoais quase não se relacionam com o resto.
>
> **Decisões:**
>
> | Decisão | Situação |
> |---|---|
> | Codificar cada serviço como "tem" (1) ou "não tem" (0), deixando a falta de internet só em `InternetService` | ◐ D10: sugestão; a ABT traz as duas versões, seleção final na modelagem |
> | Nos pares repetidos, manter a variável de maior IV como "Boa" | Decidido; aplicado na seção 8 |
> | Criar a mensalidade relativa ao pacote (preço ÷ média do mesmo tipo de internet) | A testar no notebook de variáveis |
> | `Contract` × `tenure` (0,67) e `MonthlyCharges` × `TotalCharges` (0,64) | Deixar para a seleção de variáveis |

## 7. Associação de cada variável com o churn

**O que faço:** meço, para cada variável, **se** ela se relaciona com o churn e **com que força**.
É a primeira análise que olha para o alvo, e é ela que alimenta, junto com a seção 6, a avaliação de cada variável (seção 8).

**Por que uso três medidas para cada variável?**

| Medida | Pergunta que responde | Limitação |
|---|---|---|
| **Teste de hipótese** (p-valor) | A relação existe, ou pode ser acaso? | Com 5.634 clientes, até efeitos minúsculos ficam "significativos" |
| **Tamanho do efeito** | A relação é fraca ou forte? | Cada tipo de variável tem a sua medida; não dá para comparar todas |
| **Information Value (IV)** | Quanto a variável separa quem sai de quem fica? | Mede uma variável por vez, sem considerar as outras |

> **Resumo: qual teste para qual tipo de variável (seção 3)**
>
> | Tipo da variável | Teste | Tamanho do efeito | Teoria (apêndice) |
> |---|---|---|---|
> | Qualitativa (nominal ou binária) | Qui-quadrado de independência | V de Cramér | A.1 e A.4 |
> | Qualitativa ordinal (`Contract`) | Qui-quadrado + teste de tendência | Spearman (ρ) | A.2 e A.5 |
> | Quantitativa | Mann-Whitney | Correlação rank-bisserial | A.6 |
> | Todas | — | Information Value e WoE | A.7 |
>
> O alvo `Churn` é qualitativo binário (saiu ou ficou). Por isso, todos os testes desta seção comparam **dois grupos**:
> quem saiu e quem ficou. O qui-quadrado, o V e o Spearman já foram apresentados na seção 6; aqui, mostro como se aplicam
> ao churn.

> **Perguntas para guiar**
> - Quais variáveis têm relação com o churn, e quais só parecem ter por causa do tamanho da amostra?
> - Entre as que têm relação, quais separam bem quem sai de quem fica?
> - Alguma variável separa bem **demais**, a ponto de ser suspeita?

In [ ]:
# 7.1 Associação de cada variável com o churn: teste, tamanho do efeito e Information Value (IV)
y = hist_raw["Churn"].eq("Yes").astype(int)
TAXA_GERAL = y.mean()
CORR_GRAFIA = {"yes": "Yes", "no": "No"}                   # grafia de PhoneService padronizada só para a análise

def iv_woe(grupos: pd.Series, y: pd.Series) -> float:
    """Information Value: soma de (%churn − %não churn) × ln(%churn ÷ %não churn) em cada grupo."""
    t = pd.crosstab(grupos, y)
    p1 = (t[1] / t[1].sum()).clip(lower=1e-4)
    p0 = (t[0] / t[0].sum()).clip(lower=1e-4)
    return float(((p1 - p0) * np.log(p1 / p0)).sum())

def faixa_texto(intervalo) -> str:
    """Intervalo do pandas, como (94.25, 102.65], escrito no formato brasileiro: 94,2 a 102,7."""
    return f"{virg(max(intervalo.left, 0), 1)} a {virg(intervalo.right, 1)}"

linhas = []
for col in QUALI:
    s = hist_raw[col].replace(CORR_GRAFIA)
    tab = pd.crosstab(s, y)
    qui2, p, gl, _ = stats.chi2_contingency(tab)
    v_cramer = np.sqrt(qui2 / (len(s) * (min(tab.shape) - 1)))
    taxa = y.groupby(s).mean()
    extra = ""
    if meta.loc[col, "tipo_estatistico"] == "qualitativa ordinal":        # ordinal: testa a tendência
        ordem = {c: k for k, c in enumerate(meta.loc[col, "dominio_esperado"])}
        rho, p_tend = stats.spearmanr(s.map(ordem), y)
        extra = f"tendência: Spearman {virg(rho, 3)} (p {'< 0,001' if p_tend < 0.001 else virg(p_tend, 3)})"
    linhas.append({"variavel": col, "tipo_estatistico": meta.loc[col, "tipo_estatistico"],
                   "teste": "Qui-quadrado", "estatistica": qui2, "p_valor": p,
                   "efeito": v_cramer, "medida_efeito": "V de Cramér", "IV": iv_woe(s, y),
                   "churn_min_pct": 100 * taxa.min(), "churn_max_pct": 100 * taxa.max(),
                   "categoria_maior_churn": taxa.idxmax(), "observacao": extra})

for col in QUANT:
    x = pd.to_numeric(hist_raw[col], errors="coerce")
    ok = x.notna()
    x1, x0 = x[ok & (y == 1)], x[ok & (y == 0)]
    u, p = stats.mannwhitneyu(x1, x0)
    r_rb = 1 - 2 * u / (len(x1) * len(x0))                             # correlação rank-bisserial (efeito)
    decis = pd.qcut(x[ok], 10, duplicates="drop")
    taxa = y[ok].groupby(decis, observed=True).mean()
    linhas.append({"variavel": col, "tipo_estatistico": meta.loc[col, "tipo_estatistico"],
                   "teste": "Mann-Whitney", "estatistica": u, "p_valor": p,
                   "efeito": abs(r_rb), "medida_efeito": "rank-bisserial (|r|)", "IV": iv_woe(decis, y[ok]),
                   "churn_min_pct": 100 * taxa.min(), "churn_max_pct": 100 * taxa.max(),
                   "categoria_maior_churn": faixa_texto(taxa.idxmax()),
                   "observacao": f"mediana {virg(x0.median(), 1)} (ficou) x {virg(x1.median(), 1)} (saiu)"})

def faixa_iv(iv):
    return ("sem poder" if iv < 0.02 else "fraco" if iv < 0.10 else "médio" if iv < 0.30
            else "forte" if iv < 0.50 else "muito forte (conferir vazamento)")

assoc = pd.DataFrame(linhas)
assoc["poder_iv"] = assoc["IV"].apply(faixa_iv)
assoc["significativo_5pct"] = assoc["p_valor"] < 0.05
assoc = assoc.sort_values("IV", ascending=False).reset_index(drop=True)
display(assoc.round(4))

> **Como ler a tabela 7.1**
>
> Cada linha é uma variável, ordenada do maior para o menor IV. As colunas seguem a lógica das três medidas da 7.0:
>
> | Coluna | O que é | Exemplo na tabela |
> |---|---|---|
> | `teste` e `estatistica` | Teste escolhido pelo tipo da variável (qui-quadrado para as qualitativas, Mann-Whitney para as quantitativas) e o valor calculado (χ² ou U) | `Contract`: χ² = 919,4 |
> | `p_valor` e `significativo_5pct` | Chance de ver uma diferença desse tamanho se não houvesse relação nenhuma. Abaixo de 0,05, rejeito H0 | 0 (arredondado) para quase todas; `gender`: 0,42, sem relação |
> | `efeito` e `medida_efeito` | Força da relação: V de Cramér (qualitativas, A.4) ou rank-bisserial (quantitativas, A.6), de 0 a 1 | `tenure`: 0,47; `MultipleLines`: 0,04 |
> | `IV` e `poder_iv` | Quanto a variável separa quem sai de quem fica, na mesma escala para todas (A.7) | `Contract`: 1,17 (muito forte) |
> | `churn_min_pct` e `churn_max_pct` | Menor e maior taxa de churn entre as categorias (ou entre os decis, nas quantitativas). Quanto mais distantes, mais a variável separa | `Contract`: de 3,3% a 42,7%; `gender`: de 26,2% a 27,2% |
> | `categoria_maior_churn` | Categoria ou faixa onde o churn é mais alto: onde está o risco | `PaymentMethod`: Electronic check (45,1%) |
> | `observacao` | Informação extra: tendência (ordinal, A.5) ou medianas de quem ficou e de quem saiu (quantitativas) | `tenure`: mediana 37 x 10 |
>
> **Como as colunas se combinam** (leia da esquerda para a direita):
> 1. **O p-valor responde "existe relação?".** Aqui, 17 de 20 variáveis dizem que sim, porque, com 5.634 clientes, quase tudo é significativo.
> 2. **O efeito e o IV respondem "a relação importa?".** É aqui que as variáveis se separam de verdade.
> 3. **A faixa de churn (mínimo a máximo) traduz o IV em linguagem de negócio.** Um IV alto sempre vem com uma faixa larga.
>
> **Três padrões para reconhecer:**
>
> | Padrão | Exemplo | Leitura |
> |---|---|---|
> | p pequeno + efeito alto + IV alto | `Contract`, `tenure`, `OnlineSecurity` | Variável forte |
> | p pequeno + efeito baixo + IV baixo | `MultipleLines` (p = 0,003, V = 0,04, IV = 0,01) | Relação real, mas pequena demais para ajudar: significativo não é o mesmo que útil |
> | p grande + efeito quase zero | `gender`, `PhoneService`, `MonthlyIncome` | Sem relação com o churn |
>
> **Sobre o rótulo "muito forte (conferir vazamento)":** é um alerta automático para todo IV acima de 0,50, não uma
> conclusão. As 7 variáveis com esse rótulo foram conferidas na 7.4 e são legítimas, porque existem antes do cancelamento.
> O vazamento verdadeiro aparece na 7.3.

In [ ]:
# 7.2 Ranking pelo Information Value: o que separa quem sai de quem fica
FAIXAS_IV = [(0, 0.02, "sem poder"), (0.02, 0.10, "fraco"), (0.10, 0.30, "médio"),
             (0.30, 0.50, "forte"), (0.50, 1.48, "muito forte")]
FUNDOS = ["#ffffff", "#f8f8f8", "#f1f1f1", "#eaeaea", "#e3e3e3"]   # cinza crescente: não compete com as cores das barras
COR_GRUPO = {"Contrato": DESTAQUE, "Serviços": COR, "Financeiro": "#7a8a91", "Cliente": "#b8c4c9"}

a = assoc.sort_values("IV").merge(meta[["grupo"]], left_on="variavel", right_index=True)
fig, ax = plt.subplots(figsize=(11, 8))
for (ini, fim, rot), fundo in zip(FAIXAS_IV, FUNDOS):
    ax.axvspan(ini, fim, color=fundo, zorder=0)
    if ini > 0:                                   # "sem poder" é estreita demais para o nome: fica no eixo x
        ax.axvline(ini, color="#a0a0a0", linewidth=0.8, zorder=1)
        ax.text((ini + min(fim, 1.25)) / 2, len(a) - 0.2, rot, ha="center", va="bottom", fontsize=9,
                color="#555555", fontweight="bold")
ax.barh(a["variavel"], a["IV"], color=a["grupo"].map(COR_GRUPO), height=0.65, zorder=2)

# rótulo direto: o IV e o que ele significa em churn (menor e maior taxa entre categorias ou faixas)
for _, r in a.iterrows():
    ax.text(r["IV"] + 0.01, r["variavel"],
            f"{virg(r['IV'], 2)}  ·  churn de {virg(r['churn_min_pct'], 0)}% a {virg(r['churn_max_pct'], 0)}%",
            va="center", fontsize=8.5, color=TEXTO, zorder=3)

# o texto do feedback ficaria fora da escala: nota no espaço vazio, em vez de barra
ax.text(0.62, 1.2, "Fora do gráfico: o texto do feedback (CustomerFeedback)\nteria IV de 11,4, quase dez vezes o do Contract.\n"
        "Não é poder preditivo, é vazamento (7.3).", fontsize=8.5, color=TEXTO, va="center",
        bbox=dict(facecolor="white", edgecolor="#a0a0a0", boxstyle="round,pad=0.5"), zorder=4)

ax.set_xlim(0, 1.48)
ax.set_ylim(-0.7, len(a) + 0.6)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
ax.set_xlabel("Information Value (IV)  ·  abaixo de 0,02: sem poder preditivo")
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=g) for g, c in COR_GRUPO.items()], title="grupo da variável",
          loc="center left", bbox_to_anchor=(0.62, 0.30), frameon=False, fontsize=9, title_fontsize=9)
fig.suptitle("Contrato, tempo de casa e serviços de proteção são o que mais separa quem sai de quem fica;\n"
             "dados pessoais têm poder médio, e gênero, renda e telefonia não ajudam",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, a["variavel"].tolist()[::-1] + ["CustomerFeedback", "Churn"])
plt.show()

> **Como ler o gráfico 7.2**
> - **Comprimento da barra:** o IV, ou seja, quanto a variável separa quem sai de quem fica. O fundo escurece com o
>   poder preditivo: branco (sem poder), depois fraco, médio, forte e muito forte, com o nome de cada faixa no topo.
> - **Texto ao lado da barra:** o IV e a menor e a maior taxa de churn dentro da variável. É a tradução do IV para o
>   negócio: em `Contract`, o churn vai de 3% (bienal) a 43% (mensal); em `gender`, de 26% a 27%, praticamente igual.
> - **Como calculei a faixa de churn** (colunas `churn_min_pct` e `churn_max_pct` da tabela 7.1):
>   - **Qualitativas:** para cada categoria, taxa de churn = clientes que saíram ÷ total de clientes da categoria.
>     A faixa vai da categoria com a menor taxa até a de maior taxa.
>     Exemplo, `Contract`: bienal 44 ÷ 1.347 = 3,3%; mensal 1.322 ÷ 3.099 = 42,7%.
>   - **Quantitativas:** divido os clientes em 10 faixas de mesmo tamanho (decis, cerca de 560 clientes cada) e
>     calculo a taxa de churn de cada faixa da mesma forma. São as mesmas faixas usadas no cálculo do IV.
>     Exemplo, `tenure`: 58% de churn na faixa de 0 a 2 meses e 4% na faixa de clientes mais antigos.
>   - **Referência:** a taxa geral da carteira é 26,7%. Uma variável útil tem categorias bem acima **e** bem abaixo dela;
>     uma variável inútil tem todas perto dela.
> - **Cor:** o grupo da variável. O topo é todo laranja e azul-escuro (contrato e serviços); os dados pessoais (cinza-claro)
>   ficam no meio e embaixo.
> - **O que procurar:** variáveis com barra longa **e** faixa de churn larga são as candidatas mais fortes para o modelo.
>   Barras quase invisíveis com faixa estreita (25% a 29%) não ajudam.
> - **Lembrando a seção 6:** parte do IV do streaming e da mensalidade vem do tipo de internet (η de 0,83 a 0,91). As
>   barras não se somam: duas variáveis repetidas não valem o dobro.
> - **A nota no canto:** o texto do feedback não aparece como barra porque o IV dele (11,4) achataria todas as outras.
>   Ele não é uma variável forte; é vazamento.

In [ ]:
# 7.2b Taxa de churn por categoria: onde está o risco dentro de cada variável qualitativa
TAXA_PCT = 100 * TAXA_GERAL
MARGEM = 3                                                         # até 3 pontos da média: "perto da média"
def cor_taxa(t):
    return DESTAQUE if t > TAXA_PCT + MARGEM else COR if t < TAXA_PCT - MARGEM else CINZA

iv = assoc.set_index("variavel")["IV"]
PAINEIS_QUALI = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling",
                 "OnlineSecurity", "TechSupport", "OnlineBackup", "DeviceProtection",
                 "SeniorCitizen", "Dependents", "Partner", "gender"]
fig, axes = plt.subplots(3, 4, figsize=(15, 10.5))
for ax, col in zip(axes.flat, PAINEIS_QUALI):
    s = hist_raw[col].replace(CORR_GRAFIA)
    ordem = [c for c in meta.loc[col, "dominio_esperado"] if c in set(s)]
    taxa = (100 * y.groupby(s).mean()).reindex(ordem)[::-1]
    n = s.value_counts().reindex(ordem)[::-1]
    ax.barh(taxa.index, taxa.values, color=[cor_taxa(t) for t in taxa], height=0.6)
    for cat, t in taxa.items():
        ax.text(t + 1, cat, f"{virg(t, 0)}%  (n = {virg(n[cat], 0)})", va="center", fontsize=8, color=TEXTO,
                bbox=dict(facecolor="white", edgecolor="none", pad=0.5), zorder=3)
    ax.axvline(TAXA_PCT, color="#666666", linestyle="--", linewidth=1)
    ax.set_xlim(0, 75)
    ax.set_xticks([])
    ax.spines[["bottom", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0, labelsize=8.5)
    ax.set_title(col, pad=16)
    ax.text(0, 1.02, f"IV {virg(iv[col], 2)}", transform=ax.transAxes, fontsize=8.5, color="#666666", style="italic")
axes.flat[0].text(TAXA_PCT + 1, -0.75, f"média {virg(TAXA_PCT, 1)}%", fontsize=8, color="#666666")
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=DESTAQUE, label=f"acima da média (mais de {virg(TAXA_PCT + MARGEM, 0)}%)"),
                    Patch(color=CINZA, label="perto da média (±3 pontos)"),
                    Patch(color=COR, label=f"abaixo da média (menos de {virg(TAXA_PCT - MARGEM, 0)}%)")],
           loc="upper left", bbox_to_anchor=(0.01, 0.945), ncol=3, frameon=False, fontsize=9)
fig.suptitle("O risco se concentra no contrato mensal, na fibra, no cheque eletrônico e em quem não tem proteção ou suporte;\n"
             "em gender, as barras ficam na média: a variável não separa quem sai",
             x=0.01, y=0.995, ha="left", fontsize=12, fontweight="bold")
rodape(fig, PAINEIS_QUALI + ["Churn"])
fig.subplots_adjust(top=0.86, hspace=0.5)
plt.show()

In [ ]:
# 7.2c Taxa de churn por faixa: como o risco muda ao longo de cada variável quantitativa
FAIXAS_QUANT = {"tenure": range(0, 78, 6), "MonthlyCharges": range(10, 130, 10),
                "TotalCharges": range(0, 10000, 1000), "MonthlyIncome": range(0, 11000, 1000)}
UNIDADE = {"tenure": "meses", "MonthlyCharges": "R$", "TotalCharges": "R$", "MonthlyIncome": "R$"}
TAXAS_FAIXA = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, FAIXAS_QUANT):
    x = pd.to_numeric(hist_raw[col], errors="coerce")
    ok = x.notna()
    faixa = pd.cut(x[ok], bins=list(FAIXAS_QUANT[col]), right=False)
    taxa = 100 * y[ok].groupby(faixa, observed=True).mean()
    TAXAS_FAIXA[col] = taxa
    rot = [f"{virg(f.left, 0)}" for f in taxa.index]
    ax.bar(range(len(taxa)), taxa.values, color=[cor_taxa(t) for t in taxa], width=0.8)
    for i, t in enumerate(taxa):
        ax.text(i, t + 1, f"{virg(t, 0)}%", ha="center", fontsize=7.5, color=TEXTO)
    ax.axhline(TAXA_PCT, color="#666666", linestyle="--", linewidth=1)
    ax.set_xticks(range(len(taxa)), rot, fontsize=8)                     # rótulos curtos: sempre na horizontal
    ax.set_xlabel(f"início da faixa ({UNIDADE[col]})" if UNIDADE[col] else "início da faixa")
    ax.set_ylim(0, 75)
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.set_title(col, pad=16)
    ax.text(0, 1.02, f"IV {virg(iv[col], 2)}  ·  cada barra = % de churn na faixa",
            transform=ax.transAxes, fontsize=8, color="#666666", style="italic")
axes.flat[0].text(len(FAIXAS_QUANT["tenure"]) - 2.5, TAXA_PCT + 2, f"média {virg(TAXA_PCT, 1)}%", fontsize=8,
                  color="#666666", ha="right")
t_ten = TAXAS_FAIXA["tenure"]
fig.suptitle(f"O churn despenca com o tempo de casa ({virg(t_ten.iloc[0], 0)}% nos primeiros 6 meses, "
             f"{virg(t_ten.iloc[-2:].max(), 0)}% ou menos depois de 5 anos),\n"
             "é maior nas mensalidades de R$ 70 a 110 (pacotes com fibra), e a renda não muda o risco",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, list(FAIXAS_QUANT) + ["Churn"])
plt.show()

> **Como ler os gráficos 7.2b e 7.2c**
>
> **7.2b (qualitativas):** um painel por variável, ordenados pelo IV.
> - Cada barra é a **taxa de churn da categoria**: clientes que saíram ÷ total da categoria. Ao lado, a taxa e o número de clientes (n).
> - A **linha tracejada** é a taxa da carteira toda (26,7%).
> - **Cor:** laranja = acima da média (mais de 30%); azul = abaixo (menos de 24%); cinza = perto da média (±3 pontos).
> - **O que procurar:** variáveis com barras laranja **e** azuis bem longe da linha separam bem quem sai. Em `gender`, as
>   duas barras ficam em cinza, na linha: a variável não separa.
> - **Repare nos serviços:** "No internet service" tem 7% de churn em todos os painéis. É o mesmo grupo de 1.233 clientes,
>   e é por isso que esses serviços parecem tão fortes (seção 6).
>
> **7.2c (quantitativas):** cada barra é a taxa de churn numa faixa de valores (por exemplo, `tenure` de 6 em 6 meses),
> com as mesmas cores e a mesma linha de referência.
> - **O que procurar:** a **forma** da relação. Em `tenure`, o churn cai quase sempre, rápido no começo e devagar depois.
>   Em `MonthlyCharges`, não é uma reta: o risco é baixo até R$ 30 (só telefonia), sobe a partir de R$ 70 (fibra) e cai
>   de novo na faixa mais alta. Em `MonthlyIncome`, todas as barras ficam na média.
> - **Por que isso importa para o modelo:** uma regressão logística supõe que o efeito de cada variável no logit é uma
>   reta. Relações em curva, como a do `tenure`, ou com subidas e descidas, como a da mensalidade, pedem transformações
>   (log, faixas) no notebook de variáveis.

In [ ]:
# 7.2d Distribuição de quem ficou e de quem saiu: o que o teste de Mann-Whitney compara
mw = assoc.set_index("variavel")
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, col in zip(axes.flat, ["tenure", "MonthlyCharges", "TotalCharges", "MonthlyIncome"]):
    x = pd.to_numeric(hist_raw[col], errors="coerce")
    grupos_mw = [x[(y == 0) & x.notna()], x[(y == 1) & x.notna()]]
    caixa = ax.boxplot(grupos_mw, vert=False, widths=0.55, patch_artist=True, showfliers=False,
                       medianprops=dict(color="white", linewidth=2))
    for patch, cor in zip(caixa["boxes"], [COR, DESTAQUE]):
        patch.set_facecolor(cor); patch.set_edgecolor(cor)
    for i, g in enumerate(grupos_mw, start=1):
        ax.text(g.median(), i + 0.38, f"mediana {virg(g.median(), 0)}", ha="center", fontsize=8.5, color=TEXTO)
    ax.set_yticks([1, 2], ["ficou", "saiu"])
    ax.set_ylim(0.4, 2.75)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 0)))
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    ax.set_title(col, pad=16)
    p = mw.loc[col, "p_valor"]
    ax.text(0, 1.02, f"rank-bisserial |r| = {virg(mw.loc[col, 'efeito'], 2)}  ·  p {'< 0,001' if p < 0.001 else '= ' + virg(p, 2)}",
            transform=ax.transAxes, fontsize=8.5, color="#666666", style="italic")
fig.suptitle("Quem sai tem bem menos tempo de casa e paga mensalidade maior;\n"
             "na renda, as duas caixas são praticamente iguais (|r| = 0,02)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["tenure", "MonthlyCharges", "TotalCharges", "MonthlyIncome", "Churn"])
plt.show()

> **Como ler o gráfico 7.2d**
> - **Duas caixas por variável:** azul = clientes que ficaram; laranja = clientes que saíram. Cada caixa vai do 1º ao
>   3º quartil, e a linha branca é a mediana (escrita acima da caixa).
> - **Sob o título de cada painel:** o tamanho do efeito do Mann-Whitney (rank-bisserial, A.6) e o p-valor da tabela 7.1.
> - **O que procurar:** o quanto as caixas se sobrepõem. É exatamente o que o Mann-Whitney mede: se um cliente sorteado
>   entre os que saíram tende a ter valor maior ou menor que um sorteado entre os que ficaram.
>   - `tenure`: as caixas quase não se sobrepõem (|r| = 0,47): quem sai é bem mais novo de casa;
>   - `MonthlyCharges`: quem sai paga mais (mediana de 79,8 contra 64,3);
>   - `TotalCharges`: quem sai acumulou menos, porque está há menos tempo (seção 6);
>   - `MonthlyIncome`: as caixas são praticamente iguais. A renda não separa quem sai.

In [ ]:
# 7.3 O texto do feedback: ele "conta" o desfecho? (teste de vazamento)
TERMOS_CANCELAMENTO = r"decided to churn|decided to cancel|recently canceled|recently cancelled|have canceled|switched to|decided to switch|decided to leave"
fb = hist_raw["CustomerFeedback"].str.lower()
menciona = fb.str.contains(TERMOS_CANCELAMENTO, regex=True).astype(int)

tab = pd.crosstab(menciona.map({1: "menciona cancelamento", 0: "não menciona"}),
                  y.map({1: "saiu", 0: "ficou"}), margins=True, margins_name="Total")
display(tab)
taxa_menciona = y[menciona == 1].mean()
cobertura = menciona[y == 1].mean()
print(f"Entre quem menciona cancelamento, {virg(100 * taxa_menciona, 1)}% saíram.")
print(f"Entre quem saiu, {virg(100 * cobertura, 1)}% mencionam cancelamento no texto.")
print(f"IV de uma única regra de texto: {virg(iv_woe(menciona, y), 2)} (Contract, a melhor variável, tem {virg(assoc['IV'].max(), 2)})")
print(f"Na inferência, {virg(100 * inf_raw['CustomerFeedback'].str.lower().str.contains(TERMOS_CANCELAMENTO, regex=True).mean(), 1)}% dos textos mencionam cancelamento.")

print("\nExemplos (dois que saíram e dois que ficaram):")
for rot, mask in [("saiu", y == 1), ("ficou", y == 0)]:
    for t in hist_raw.loc[mask, "CustomerFeedback"].head(2):
        print(f"  [{rot}] {t[:220]}...")

In [ ]:
# 7.3b Mais evidências de que o texto foi escrito depois do desfecho (e a partir do próprio registro do cliente)
# 1) cada expressão de cancelamento, sozinha: quantos textos e quantos desses clientes saíram
EXPRESSOES = ["decided to churn", "decided to cancel", "decided to switch", "switched to",
              "recently canceled", "decided to leave", "had to churn", "made the decision to churn"]
ev = pd.DataFrame([{"expressao": e, "textos": int(fb.str.contains(e).sum()),
                    "pct_que_saiu": 100 * y[fb.str.contains(e)].mean()} for e in EXPRESSOES])
display(ev.round(1))

# 2) um "modelo" de uma linha: prever churn quando o texto tem a expressão
TERMOS_AMPLOS = TERMOS_CANCELAMENTO + r"|had to churn|made the decision to churn|decision to churn"
regra = fb.str.contains(TERMOS_AMPLOS, regex=True).astype(int)
acerto_regra = (regra == y).mean()
acerto_chute = 1 - y.mean()                                        # chutar "ficou" para todos
print(f"Regra de uma linha sobre o texto: acerta {virg(100 * acerto_regra, 1)}% dos clientes.")
print(f"Chutar 'ficou' para todo mundo acerta {virg(100 * acerto_chute, 1)}%.")

# 3) o texto repete os dados do próprio registro: cita a mensalidade exata do cliente?
cita_mensalidade = np.mean([m in t for m, t in zip(hist_raw["MonthlyCharges"], hist_raw["CustomerFeedback"])])
print(f"{virg(100 * cita_mensalidade, 1)}% dos textos citam a mensalidade exata do cliente (ex.: '$73.9').")

# 4) quem saiu e não foi pego pela regra original também escreve no passado
print("\nClientes que saíram sem as expressões da 7.3 (trechos):")
for t in hist_raw.loc[(y == 1) & (menciona == 0), "CustomerFeedback"].head(3):
    i = max(t.lower().find("churn"), 0)
    print(f"  ...{t[max(i - 80, 0):i + 60]}...")

In [ ]:
# 7.3c O vazamento em uma figura: quem menciona cancelamento no texto saiu em 100% dos casos
grupos = pd.DataFrame({
    "grupo": ["menciona cancelamento no texto", "não menciona", "carteira toda"],
    "n": [int(menciona.sum()), int((menciona == 0).sum()), len(y)],
    "saiu": [100 * y[menciona == 1].mean(), 100 * y[menciona == 0].mean(), TAXA_PCT],
})
grupos["ficou"] = 100 - grupos["saiu"]
fig, ax = plt.subplots(figsize=(11, 3.6))
yy = np.arange(len(grupos))[::-1]
ax.barh(yy, grupos["saiu"], color=DESTAQUE, height=0.55)
ax.barh(yy, grupos["ficou"], left=grupos["saiu"], color="#d6dee2", height=0.55)
for y0, (_, r) in zip(yy, grupos.iterrows()):
    if r["saiu"] > 8:
        ax.text(r["saiu"] / 2, y0, f"saiu {virg(r['saiu'], 1)}%", ha="center", va="center", color="white",
                fontsize=10, fontweight="bold")
    else:
        ax.text(r["saiu"] + 1, y0, f"saiu {virg(r['saiu'], 1)}%", va="center", color=DESTAQUE, fontsize=10,
                fontweight="bold")
    if r["ficou"] > 8:
        ax.text(r["saiu"] + r["ficou"] / 2, y0, f"ficou {virg(r['ficou'], 1)}%", ha="center", va="center",
                color=TEXTO, fontsize=10)
ax.set_yticks(yy, [f"{g}\n({virg(n, 0)} clientes)" for g, n in zip(grupos["grupo"], grupos["n"])], fontsize=9.5)
ax.set_xlim(0, 100)
ax.set_xticks([])
ax.spines[["bottom", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.suptitle("Quem menciona o cancelamento no texto saiu em 100% dos casos; quem não menciona, em só 2%.\n"
             "O texto não prevê o churn: ele registra um churn que já aconteceu",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["CustomerFeedback", "Churn"])
plt.show()

> **Como ler as saídas da 7.3, da 7.3b e o gráfico 7.3c**
>
> **Tabela da 7.3 (menciona cancelamento × saiu ou ficou)**
> - As linhas separam os textos que contêm alguma expressão de cancelamento ("decided to churn", "switched to"...)
>   dos que não contêm. As colunas mostram quantos desses clientes ficaram e quantos saíram.
> - **O número que importa é o 0** na linha "menciona cancelamento", coluna "ficou". Nenhum cliente que ficou usa essas
>   expressões. Por isso, "100% saíram" entre os que mencionam.
> - **A cobertura de 94,5%** diz que quase todos os que saíram (1.421 de 1.504) escreveram a decisão no texto.
> - **O IV de 11,38** compara essa regra simples com as variáveis da 7.1: é quase dez vezes o IV do `Contract`.
> - **Os 24,4% da inferência** mostram que o mesmo padrão está nos clientes que vamos pontuar.
> - **Os exemplos** mostram a diferença no próprio texto: quem saiu conta a decisão ("I have decided to churn"); quem
>   ficou descreve o serviço no presente ("I have been satisfied").
>
> **Saídas da 7.3b**
> - **Tabela de expressões:** cada linha é uma expressão testada sozinha. Todas têm 100% de clientes que saíram. Não é uma
>   palavra solta que coincide com o churn, são oito formas diferentes de contar a mesma decisão, sempre no passado.
> - **98,9% contra 73,3%:** a régua para julgar o acerto. Chutar "ficou" para todo mundo já acerta 73,3%, porque 73,3% dos
>   clientes ficaram. Qualquer modelo precisa ser comparado com esse chute, e não com zero. A regra de texto sobe para
>   98,9%: um salto que nenhuma variável legítima explica.
> - **88,7% citam a mensalidade exata:** compara o valor de `MonthlyCharges` de cada cliente com o texto dele. Se quase
>   todos repetem o valor com centavos, os textos foram montados a partir da linha da base, e não escritos livremente.
> - **Os trechos finais:** são clientes que saíram e escaparam da regra original. Os dois primeiros também contam a decisão
>   ("had to churn after only one month"). O terceiro ("to prevent further churn") é uma opinião sobre a empresa, e não
>   um relato de saída. Mostra que a regra não é perfeita, mas os casos que ela perde costumam conter a mesma informação
>   escrita de outro jeito.
>
> **Gráfico 7.3c:** cada barra é um grupo de clientes, dividido entre os que saíram (laranja) e os que ficaram (cinza).
> - **Em cima, quem menciona cancelamento:** a barra é toda laranja, 100% saíram.
> - **No meio, quem não menciona:** só 2% saíram.
> - **Embaixo, a carteira toda (26,7%)**, como referência.
> - **O que procurar:** num preditor legítimo, os grupos teriam taxas diferentes, mas nunca 100% contra 2%. Uma separação
>   perfeita assim é o sinal visual do vazamento.
>
> **Em uma frase:** a 7.3 e o gráfico 7.3c mostram que o texto separa perfeitamente quem sai de quem fica; a 7.3b mostra
> **por quê**: ele foi escrito depois da decisão e a partir dos dados do próprio cliente. A 7.3d reúne os argumentos.

### 7.3d Por que não uso o texto do feedback para prever o churn

**A tentação:** o `CustomerFeedback` é, de longe, a "melhor variável" da base. Uma regra de uma linha acerta 98,9% dos
clientes, e qualquer modelo que use o texto vai parecer excelente. Justamente por isso, ele precisa ser examinado antes
de entrar. Meus argumentos, do mais forte para o mais fraco:

> **1. O texto foi escrito depois do desfecho (argumento temporal).**
> Os clientes que saíram escrevem no **passado** e contam a decisão: "I **have decided to churn**", "I **switched to**
> another provider", "**had to churn** after only one month". Oito expressões diferentes aparecem só em clientes que
> saíram, todas com 100% de churn, e nenhuma em quem ficou. Um modelo de churn precisa prever **antes** do
> cancelamento, e esse texto só existe **depois** dele. Usá-lo é como prever o resultado de um jogo lendo a notícia do
> dia seguinte.

> **2. Na operação, o modelo não teria utilidade.**
> O objetivo do negócio é o time de vendas ligar para clientes **em risco** e oferecer retenção. Um cliente que já escreveu
> "decidi cancelar" **já foi perdido**: prever que ele vai sair não gera nenhuma ação. O modelo com texto acertaria
> justamente os casos em que não há mais nada a fazer, e erraria os que importam: quem ainda está decidindo.

> **3. É bom demais para ser verdade.**
> Uma única regra de texto tem IV de 11,4, quase dez vezes o da melhor variável legítima (`Contract`, 1,17), e acerta
> 98,9% dos clientes. Em problemas reais de churn, nenhuma informação disponível antes do cancelamento separa os grupos
> com essa perfeição. Quando um resultado é perfeito demais, a primeira hipótese deve ser vazamento, não uma descoberta.

> **4. O texto parece ter sido gerado a partir do registro completo do cliente.**
> 88,7% dos textos citam a mensalidade **exata** do cliente ("$73.9"), além do tempo de casa, do tipo de contrato e da
> forma de pagamento. Um cliente real não lembra a mensalidade com centavos. Tudo indica que os textos foram montados
> a partir da linha da base, **incluindo a coluna `Churn`**. Isso explica por que o desfecho aparece escrito neles.

> **5. O modelo aprenderia a palavra, não o comportamento.**
> Com o texto, o modelo aprende a procurar "churn" ou "switched to", e não por que os clientes saem. Ele não ensina nada
> sobre retenção (contrato, preço, suporte) e deixa de funcionar quando os textos não contiverem a decisão, que é
> exatamente a situação real: a previsão é feita sobre quem ainda não decidiu.

> **6. As métricas enganariam o negócio.**
> Um modelo com 99% de acerto faria a empresa esperar uma retenção que nunca aconteceria. A métrica honesta é a do modelo
> sem texto: menor, mas real.

**O contra-argumento, e por que não muda a decisão:** 24,4% dos textos da **inferência** também mencionam o cancelamento.
Se a avaliação do hackathon comparar a previsão com o churn real desses clientes, um modelo com texto vai pontuar muito
alto. Isso pode ser intencional: a base parece ter sido montada para testar se o grupo percebe o vazamento. Mesmo que
não seja, o modelo com texto continua inútil para o problema de negócio que o case descreve.

**Decisão do grupo (D9, 10/11):**
- **Modelo oficial sem o texto**, com métricas honestas e utilidade real para o time de vendas.
- **Mostrar o modelo com texto na apresentação, como contraexemplo:** "olhem o que acontece se usarmos o texto: 99% de
  acerto, e por que isso é uma armadilha". Identificar o vazamento é um resultado do trabalho, não uma limitação.
- **O texto continua útil para a apresentação:** ler os motivos que os clientes citam (preço, suporte, concorrência)
  ajuda a desenhar as ofertas de retenção. Só não pode entrar como variável de previsão.
- **Uso legítimo no futuro:** textos coletados **antes** do cancelamento (chamados de suporte, reclamações, pesquisas de
  satisfação, com data) poderiam virar variáveis de sentimento. Aqui não há data, então não há como separar.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. Os argumentos convencem? Qual deles vocês usariam primeiro na apresentação?
> 2. Que contra-argumento a banca pode trazer (por exemplo, "mas o modelo com texto acerta mais")? Como respondemos?
>
> **Respostas e decisões do grupo:**
>
> &nbsp;

> **Explicação**
>
> **1. Quase tudo é significativo, e por isso não confio só no p-valor.** Com 5.634 clientes, 17 das 20 variáveis
> têm p < 0,05. O que diferencia uma variável útil de uma inútil é o **tamanho do efeito** e o **IV**.
> `MultipleLines`, por exemplo, é significativa (p = 0,003), mas tem IV de 0,01: o churn varia só de 24,9% a 29,0% entre as categorias.
>
> **2. As variáveis mais fortes** (IV acima de 0,50):
>
> | Variável | IV | Leitura |
> |---|---|---|
> | `Contract` | 1,17 | Churn de 42,7% no mensal contra 3,3% no bienal; a tendência ordinal é forte (Spearman −0,40) |
> | `tenure` | 0,77 | 58,2% de churn nos dois primeiros meses; mediana de 37 meses entre quem fica e de 10 entre quem sai |
> | `OnlineSecurity`, `TechSupport` | 0,72 e 0,71 | Quem não tem esses serviços sai cerca de 42% das vezes |
> | `InternetService` | 0,63 | Fibra ótica concentra o maior churn (42,1%) |
> | `OnlineBackup`, `DeviceProtection` | 0,53 e 0,50 | Mesmo padrão dos serviços de proteção: sem o serviço, cerca de 40% saem |
>
> **Conferi o alerta de IV acima de 0,50.** Essas variáveis são legítimas: contrato, tempo de casa e serviços
> **são conhecidos antes** de o cliente cancelar. O IV alto reflete um efeito real, não vazamento.
>
> **3. Como o cliente se relaciona importa mais do que quem ele é.** No gráfico 7.2, o topo é todo de variáveis de
> contrato e serviços. Os dados pessoais (`Dependents`, `SeniorCitizen`, `Partner`) têm poder médio, e `gender` e
> `MonthlyIncome`, nenhum. Para o negócio, isso é uma boa notícia: contrato, pagamento e serviços são coisas que a
> empresa **pode mudar** com uma oferta de retenção; idade e gênero, não.
>
> **4. Parte do IV é informação repetida (seção 6).** `TotalCharges` tem o maior churn (51,7%) na faixa mais baixa, mas
> isso acontece porque o cliente é novo: ela repete `tenure`. Do mesmo modo, a mensalidade e o streaming carregam boa parte
> da informação do tipo de internet. A seção 8 junta as duas análises para decidir o que fica.
>
> **5. Sem poder preditivo** (IV abaixo de 0,02): `gender`, `PhoneService`, `MonthlyIncome` e `MultipleLines`.
> O churn é praticamente o mesmo em todas as categorias ou faixas.
>
> **6. O texto do feedback é vazamento (7.3 a 7.3d).** Uma regra de uma linha sobre o texto acerta 98,9% dos clientes
> (chutar "ficou" para todos acerta 73,3%) e tem IV de 11,4, quase dez vezes o da melhor variável. O texto foi escrito
> **depois** do cancelamento ("I have decided to churn") e a partir do próprio registro do cliente (88,7% citam a
> mensalidade exata). Fica fora do modelo de previsão, pela regra da ABT: ela só pode ter informação disponível no
> momento da previsão.
>
> **Para o negócio:** o retrato do cliente em risco já aparece aqui: contrato mensal, poucos meses de casa, fibra ótica,
> sem serviços de proteção e suporte e pagamento por cheque eletrônico (45,1% de churn). Na seção 8, transformo esses
> resultados na avaliação de cada variável.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. **Decisão principal:** entregamos o modelo com ou sem o texto do feedback? Quem pergunta à organização como a entrega será avaliada?
> 2. Alguém quer fazer uma análise de sentimento ou dos motivos citados no texto, só para a apresentação (sem entrar no modelo)?
> 3. Descartamos já `gender`, `PhoneService`, `MonthlyIncome` e `MultipleLines`, ou deixamos a seleção de variáveis confirmar?
> 4. Que ofertas de retenção faz sentido propor para o perfil de risco (contrato mensal, fibra, sem proteção, cheque eletrônico)?
>
> **Respostas e decisões do grupo:**
>
> - **1 (D9, 10/11):** modelo oficial **sem** o texto; o com texto só como contraexemplo (no 05, a AUC salta para 0,993). Falta a organização dizer como a entrega é avaliada.
> - **3 (D12 e D15):** ◐ sugestão de descartar; ficam na ABT e não entram na seleção sugerida do 05.

> **Resumo da seção 7 e decisões**
>
> **O que encontrei:** taxa geral de churn de 26,7%. 12 variáveis com IV ≥ 0,30, lideradas por contrato, tempo de casa
> e serviços de proteção; 4 sem poder preditivo. O perfil de risco é contrato mensal, poucos meses de casa, fibra ótica,
> sem proteção e suporte e pagamento por cheque eletrônico. O texto do feedback é vazamento.
>
> **Decisões:**
>
> | Decisão | Situação |
> |---|---|
> | Tirar o `CustomerFeedback` do modelo de previsão | ☑ D9 (10/11): fora da ABT; texto em `gold_churn_texto` para o modelo com texto. Reavaliar se a organização disser que o texto é anterior ao cancelamento |
> | Mostrar o modelo com texto como contraexemplo na apresentação | ☑ D9 (10/11): modelo com texto a partir da `gold_churn_texto` |
> | Descartar já as 4 variáveis sem poder (`gender`, `MonthlyIncome`, `PhoneService`, `MultipleLines`) | ◐ D12 e D15: sugestão de descartar; ficam na ABT, seleção final na modelagem |
> | Usar o perfil de risco para desenhar as ofertas de retenção | Para a apresentação |

## 8. Avaliação de cada variável

**O que faço:** junto o que vi nas seções 4 a 7 e dou um veredito para cada variável: **Boa**, **Atenção** ou **Descartar**.
Também registro o critério que se aplicou, a justificativa com os números e as ideias de variáveis derivadas, que vão
guiar a criação de variáveis e a seleção mais adiante.

**De onde vem cada informação:**

| Informação | Seção |
|---|---|
| Valores inválidos | 4 (qualidade) |
| Repetição entre variáveis (associação ≥ 0,8, nos 190 pares da matriz) | 6 (associação entre explicativas) |
| Poder preditivo (IV) e vazamento | 7 (associação com o churn) |

> **Dois limites de associação, com papéis diferentes**
>
> | Limite | Onde uso | Por quê |
> |---|---|---|
> | 0,5 | Tabela de pares da 6.1 | Associação forte: vale **olhar**, para entender de onde vem (6.3) |
> | 0,8 | Veredito da 8.1 | Uma variável quase determina a outra: aí, sim, é repetição que **pesa** no veredito |
>
> O 0,8 é uma convenção usual para multicolinearidade forte, e não uma regra fixa. Entre 0,5 e 0,8, a decisão fica para a
> seleção de variáveis.

> **Critérios do veredito** (aplicados nesta ordem; vale o primeiro que se aplica)
>
> | Ordem | Veredito | Quando |
> |---|---|---|
> | 1 | **Descartar** | Identificador ou vazamento (texto do feedback) |
> | 2 | **Descartar** | IV abaixo de 0,02: não separa quem sai de quem fica |
> | 3 | **Atenção** | Tem poder preditivo, mas precisa de tratamento: valores inválidos ou repete outra variável mais forte (associação ≥ 0,8) |
> | 4 | **Boa** | Tem poder preditivo, está limpa e não repete outra variável |
>
> - **"Descartar" vale para o modelo na forma original.** Uma variável descartada ainda pode entrar numa derivada
>   (por exemplo, `PhoneService` no total de serviços contratados).
> - **"Atenção" não quer dizer variável fraca.** `tenure` é a segunda mais forte da base; só precisa de tratamento antes de entrar.
> - **Quando duas variáveis se repetem**, a de maior IV fica como "Boa", e a outra recebe "Atenção".

In [ ]:
# 8.1 Avaliação de cada variável: Boa, Atenção ou Descartar, com o critério aplicado, a justificativa e ideias de derivadas
POTENCIAL = {
    "customerID": "Nenhum. Serve só de chave para juntar as bases e montar o arquivo de entrega",
    "gender": "Nenhum esperado",
    "SeniorCitizen": "Combinar com Partner e Dependents para identificar idoso que mora sozinho",
    "Partner": "sem_familia: sem parceiro(a) e sem dependentes",
    "Dependents": "sem_familia: sem parceiro(a) e sem dependentes",
    "MonthlyIncome": "Sozinha não separa; testar o peso da conta no orçamento (MonthlyCharges ÷ MonthlyIncome)",
    "tenure": "cliente_novo (até 6 meses), faixa_tenure e log(1 + tenure); recuperar os 'unknown' por TotalCharges ÷ MonthlyCharges",
    "Contract": "Usar como ordinal (0, 1, 2); interação contrato mensal × cliente novo",
    "PaperlessBilling": "Combinar com PaymentMethod (fatura digital e pagamento manual)",
    "PaymentMethod": "pagamento_automatico: transferência ou cartão automático = 1",
    "PhoneService": "Corrigir a grafia; entra em n_servicos (total de serviços contratados)",
    "MultipleLines": "Entra em n_servicos (total de serviços contratados)",
    "InternetService": "tem_fibra; mensalidade acima da média do mesmo tipo de internet",
    "OnlineSecurity": "n_protecao: soma de OnlineSecurity, OnlineBackup, DeviceProtection e TechSupport",
    "OnlineBackup": "n_protecao: soma de OnlineSecurity, OnlineBackup, DeviceProtection e TechSupport",
    "DeviceProtection": "n_protecao: soma de OnlineSecurity, OnlineBackup, DeviceProtection e TechSupport",
    "TechSupport": "n_protecao: soma de OnlineSecurity, OnlineBackup, DeviceProtection e TechSupport",
    "StreamingTV": "n_streaming: soma de StreamingTV e StreamingMovies (0 a 2)",
    "StreamingMovies": "n_streaming: soma de StreamingTV e StreamingMovies (0 a 2)",
    "MonthlyCharges": "Mensalidade ÷ média do mesmo InternetService; mensalidade por serviço contratado",
    "TotalCharges": "Preencher os nulos com MonthlyCharges × tenure; a razão TotalCharges ÷ (MonthlyCharges × tenure) indica mudança de plano",
    "CustomerFeedback": "Não entra no modelo. Serve para a apresentação: motivos citados por quem saiu (preço, suporte, concorrência)",
    "Churn": "É o alvo",
}

problemas = (qualidade[qualidade["base"] == "historico"].set_index("coluna")
             .pipe(lambda q: q["vazios"] + q["fora_do_dominio"]))
forte = pares_todos[pares_todos["associacao"] >= 0.8]           # repetição forte, nos 190 pares da matriz (6.2)

def avaliar(col: str) -> tuple:
    """Devolve (veredito, critério da tabela da 8.0 que se aplicou, justificativa com os números)."""
    m = meta.loc[col]
    if m["papel"] == "alvo":
        return "Alvo", "Alvo", "Variável resposta (Y)"
    if m["papel"] == "id":
        return "Descartar", "Identificador", "Não descreve o cliente"
    if m["papel"] == "texto":
        return "Descartar", "Vazamento", "O texto foi escrito depois do desfecho (IV de 11,4 na seção 7.3)"
    a = assoc.set_index("variavel").loc[col]
    motivos = [f"IV {virg(a['IV'], 2)} ({a['poder_iv'].replace(' (conferir vazamento)', '')})"]
    if a["IV"] < 0.02:
        return "Descartar", "IV abaixo de 0,02", motivos[0] + ": não separa quem sai de quem fica"
    criterios = []
    if problemas.get(col, 0) > 0:
        motivos.append(f"{problemas[col]} valores inválidos para tratar")
        criterios.append("Valores inválidos")
    for _, r in forte[(forte["variavel_1"] == col) | (forte["variavel_2"] == col)].iterrows():
        outra = r["variavel_2"] if r["variavel_1"] == col else r["variavel_1"]
        iv_outra = assoc.set_index("variavel").loc[outra, "IV"]
        if iv_outra >= 0.02 and iv_outra > a["IV"]:             # a outra é mais forte: esta é a repetida
            motivos.append(f"repete {outra}, que tem IV maior ({r['medida']} {virg(r['associacao'], 2)})")
            if "Repete variável mais forte" not in criterios:      # cita o critério uma vez, mesmo com várias repetições
                criterios.append("Repete variável mais forte")
    if col in SERVICOS:
        motivos.append("compartilha 'No internet service' com InternetService (6.3)")
    if criterios:
        return "Atenção", " + ".join(criterios), "; ".join(motivos)
    return "Boa", "Poder preditivo, limpa e sem repetição", "; ".join(motivos)

aval = meta[["grupo", "tipo_estatistico", "papel"]].copy()
aval[["avaliacao", "criterio", "justificativa"]] = [avaliar(c) for c in aval.index]
aval["potencial_derivada"] = aval.index.map(POTENCIAL)
aval = aval.join(assoc.set_index("variavel")["IV"])
ORDEM_AVAL = {"Boa": 0, "Atenção": 1, "Descartar": 2, "Alvo": 3}
aval = (aval.assign(_o=aval["avaliacao"].map(ORDEM_AVAL)).sort_values(["_o", "IV"], ascending=[True, False])
        .drop(columns="_o").reset_index(names="variavel"))

# resumo no formato da tabela de critérios da 8.0: veredito, critério que se aplicou e quais variáveis
resumo_aval = (aval.groupby(["avaliacao", "criterio"], sort=False)
               .agg(n_variaveis=("variavel", "size"), variaveis=("variavel", ", ".join))
               .reset_index())
display(resumo_aval)
display(aval[["variavel", "grupo", "avaliacao", "criterio", "IV", "justificativa", "potencial_derivada"]].round(3))

> **Como ler as tabelas da 8.1**
>
> **Primeira tabela (resumo):** uma linha por combinação de veredito e critério, no formato da tabela de critérios da 8.0.
> Mostra quantas variáveis caíram em cada regra e quais são. É a resposta rápida para "o que fica e o que sai".
>
> **Segunda tabela (detalhe):** uma linha por variável.
>
> | Coluna | O que é |
> |---|---|
> | `avaliacao` | O veredito: Boa, Atenção, Descartar ou Alvo |
> | `criterio` | A regra da 8.0 que decidiu o veredito (a primeira que se aplicou) |
> | `IV` | O poder preditivo da 7.1, para ordenar dentro de cada veredito |
> | `justificativa` | Os números que sustentam o veredito: IV, valores inválidos, repetição (com a medida e o valor da 6.2). Para os serviços, também a observação sobre "No internet service" |
> | `potencial_derivada` | Ideias de variáveis novas a partir desta, que vão para o notebook de variáveis |
>
> **O que procurar:** em "Atenção", a justificativa diz **o que** tratar (19 `unknown` em `tenure`; 278 nulos e a
> repetição em `TotalCharges`; a repetição dos serviços em `MonthlyCharges`). Em "Descartar", a coluna
> `potencial_derivada` mostra que algumas variáveis ainda podem ser aproveitadas dentro de uma derivada.

In [ ]:
# 8.1b Mapa do veredito: poder preditivo (IV) contra repetição (maior associação com outra variável)
assoc_max = pd.concat([pares_todos[["variavel_1", "variavel_2", "associacao", "medida"]],
                       pares_todos.rename(columns={"variavel_1": "variavel_2", "variavel_2": "variavel_1"})
                       [["variavel_1", "variavel_2", "associacao", "medida"]]])
assoc_max = (assoc_max.sort_values("associacao", ascending=False).groupby("variavel_1").head(1)
             .set_index("variavel_1"))
mapa_v = aval.set_index("variavel").join(assoc_max).dropna(subset=["IV"])     # sem customerID, texto e alvo
COR_VEREDITO = {"Boa": COR, "Atenção": DESTAQUE, "Descartar": CINZA}
LIMITE_IV, LIMITE_REP = 0.02, 0.8

fig, ax = plt.subplots(figsize=(12, 7.5))
ax.axvspan(1e-4, LIMITE_IV, color="#f3f3f3", zorder=0)
ax.axhspan(LIMITE_REP, 1.25, xmin=0, xmax=1, color="#fbeee2", zorder=0)
ax.axvline(LIMITE_IV, color="#999999", linestyle=":", linewidth=1)
ax.axhline(LIMITE_REP, color="#999999", linestyle=":", linewidth=1)
for veredito, cor in COR_VEREDITO.items():
    d = mapa_v[mapa_v["avaliacao"] == veredito]
    ax.scatter(d["IV"].clip(lower=2e-4), d["associacao"], s=90, color=cor, edgecolor="white", linewidth=1,
               zorder=3, label=f"{veredito} ({len(d)})")
invalidos = mapa_v[mapa_v["criterio"].str.contains("Valores inválidos")]
ax.scatter(invalidos["IV"], invalidos["associacao"], s=260, facecolor="none", edgecolor=TEXTO, linewidth=1.2,
           zorder=4, label="tem valores inválidos")
# rótulos: deslocamentos manuais (em pontos) para o grupo do canto superior direito, com linha guia
DESLOC = {"MonthlyCharges": (-95, 30), "InternetService": (25, 32), "TotalCharges": (-105, 8), "tenure": (30, 12),
          "StreamingMovies": (-120, -8), "StreamingTV": (-115, -26), "OnlineBackup": (-60, -48),
          "DeviceProtection": (-10, -62), "OnlineSecurity": (40, -8), "TechSupport": (30, -75),
          "gender": (8, 8), "Partner": (-48, 0), "PhoneService": (8, -10), "SeniorCitizen": (8, -10)}
for var, r in mapa_v.iterrows():
    dx, dy = DESLOC.get(var, (8, 0))
    guia = dict(arrowstyle="-", color="#aaaaaa", linewidth=0.7) if abs(dx) > 10 or abs(dy) > 10 else None
    ax.annotate(var, (max(r["IV"], 2e-4), r["associacao"]), xytext=(dx, dy), textcoords="offset points",
                fontsize=8.5, color=TEXTO, va="center", arrowprops=guia)
ax.set_xscale("log")
ax.set_xlim(1.5e-4, 3)
ax.set_ylim(0, 1.25)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 3 if v < 0.01 else 2 if v < 1 else 0)))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
ax.set_xlabel("Information Value (IV), escala logarítmica: poder de separar quem sai de quem fica (seção 7)")
ax.set_ylabel("maior associação com outra variável (seção 6)")
ax.text(1.7e-4, 1.22, "Descartar:\nsem poder (IV < 0,02)", fontsize=8.5, color="#777777", va="top")
ax.text(0.025, 1.22, "Repetição forte (≥ 0,8): Atenção se a outra variável do par tem IV maior;\nInternetService e tenure são as mais fortes dos seus pares",
        fontsize=8.5, color=DESTAQUE, va="top")
ax.text(2.8, 0.05, "Boa: forte e sem repetição", fontsize=8.5, color=COR, ha="right")
ax.legend(loc="lower left", bbox_to_anchor=(0.0, 0.08), frameon=False, fontsize=9)
fig.suptitle("A maioria das variáveis fica no canto da direita, embaixo: separa bem e não repete outra;\n"
             "MonthlyCharges e TotalCharges repetem variáveis mais fortes, e 4 variáveis não têm poder",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, list(mapa_v.index))
plt.show()

> **Como ler o mapa 8.1b**
> - **Cada ponto é uma variável explicativa.** Ficam de fora `customerID`, `CustomerFeedback` e `Churn`, que não têm IV.
> - **Eixo horizontal:** o IV (seção 7), em escala logarítmica, para que as variáveis com IV perto de zero não fiquem todas
>   empilhadas. Quanto mais à direita, mais a variável separa quem sai de quem fica.
> - **Eixo vertical:** a maior associação da variável com qualquer outra (matriz da 6.2). Quanto mais alto, mais ela repete
>   outra variável.
> - **As três regiões são as regras do veredito (8.0):**
>   - **cinza, à esquerda (IV < 0,02):** sem poder, Descartar;
>   - **laranja-claro, em cima (associação ≥ 0,8):** repetição forte. Vira "Atenção" a variável que tem o IV **menor** do
>     par. Por isso `InternetService` (o par mais forte da mensalidade) fica azul, e `MonthlyCharges`, laranja;
>   - **branco, à direita e embaixo:** forte e sem repetição, Boa.
> - **Cor do ponto:** o veredito (azul = Boa, laranja = Atenção, cinza = Descartar). **Anel preto:** a variável tem
>   valores inválidos para tratar. É o caso de `tenure`, que está em "Atenção" por isso, e não por repetir outra.
> - **O que procurar:** dentro da região laranja-clara, os pontos azuis são as variáveis mais fortes de pares repetidos
>   (`InternetService`), e os laranja, as que repetem uma variável mais forte (`MonthlyCharges`, `TotalCharges`).
>   Os dois streamings ficam logo acima de 0,8 (repetem a mensalidade, mas têm IV maior que ela); os serviços de proteção
>   ficam logo abaixo: associação alta entre eles, que vem da categoria "No internet service" (6.3), mas sem chegar à
>   repetição forte.

> **Explicação**
>
> **1. O veredito:**
>
> | Veredito | Critério | Variáveis |
> |---|---|---|
> | **Boa (13)** | Poder preditivo, limpa e sem repetição | `Contract`, `OnlineSecurity`, `TechSupport`, `InternetService`, `OnlineBackup`, `DeviceProtection`, `PaymentMethod`, `StreamingMovies`, `StreamingTV`, `PaperlessBilling`, `Dependents`, `SeniorCitizen`, `Partner` |
> | **Atenção (3)** | Valores inválidos; repete variável mais forte | `tenure` (19 `unknown` para recuperar); `MonthlyCharges` (repete `InternetService`, η = 0,91, e os streamings); `TotalCharges` (278 nulos e repete `tenure`) |
> | **Descartar (6)** | IV abaixo de 0,02; identificador; vazamento | `MultipleLines`, `MonthlyIncome`, `gender`, `PhoneService`; `customerID`; `CustomerFeedback` |
>
> **2. O que mudou por causa da seção 6.** `MonthlyCharges` tem IV forte (0,38), mas a mensalidade é quase determinada pelo
> tipo de internet e pelos serviços, que têm IV maior. Por isso recebe "Atenção". A parte útil dela é o que vai **além** do
> pacote: pagar mais ou menos que a média do próprio tipo de internet, uma das derivadas da lista.
>
> **3. "Atenção" não quer dizer variável fraca.** `tenure` é a segunda melhor variável da base; só precisa de tratamento
> antes de entrar. `MonthlyCharges` e `TotalCharges` podem entrar no modelo, mas a seleção de variáveis deve decidir
> entre elas e as variáveis que repetem.
>
> **4. Os serviços ficam como "Boa".** A associação entre eles era só a categoria "No internet service" (6.3). Na
> codificação, cada serviço vira "tem" (1) ou "não tem" (0).
>
> **5. Próximos passos.** A coluna `potencial_derivada` vira a lista de variáveis candidatas: `n_servicos`, `n_protecao`,
> `n_streaming`, `cliente_novo`, `pagamento_automatico`, `sem_familia`, `tem_fibra` e as razões de mensalidade.
> No próximo notebook, crio essas variáveis, calculo o IV de cada uma do mesmo jeito, comparo com o IV das originais
> (antes e depois) e aplico um método de seleção.
>
> **Para o negócio:** das 20 informações que a empresa tem sobre o cliente, 13 ajudam a prever o churn como estão,
> 3 ajudam depois de tratadas ou combinadas e 4 não ajudam. O texto do feedback, apesar de ser o que "mais acerta", não
> pode ser usado, porque só existe depois que o cliente já cancelou.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. `MonthlyCharges` e `TotalCharges` (Atenção por repetição): mantemos, trocamos pelas derivadas ou deixamos a seleção decidir?
> 2. Quem fica com o notebook de variáveis e seleção, e quem fica com o de cluster?
>
> **Respostas e decisões do grupo:**
>
> - **1 (D16):** ◐ sugestão: `MonthlyCharges` ou `mensalidade_relativa_pacote` como candidatas e `TotalCharges` como sugiro descartar. Nenhuma das três entrou na seleção sugerida do 05 (D19).

> **Resumo da seção 8 e decisões**
>
> **O que encontrei:** 13 variáveis boas, 3 com atenção (`tenure`, `MonthlyCharges`, `TotalCharges`) e 6 para descartar.
>
> **Decisões:**
>
> | Decisão | Situação |
> |---|---|
> | Variáveis "Boa" entram como candidatas no modelo | Decidido |
> | `tenure` entra depois de recuperar os `unknown` | Decidido |
> | `MonthlyCharges` e `TotalCharges`: manter, trocar pela derivada ou deixar a seleção decidir | ◐ D16: sugestão no 04; seleção final na modelagem |
> | Criar as derivadas da coluna `potencial_derivada` e comparar o IV antes e depois | Feito no 04 |
> | Derivadas testadas no notebook 04: candidatas `is_monthly_contract`, `no_support_with_internet`, `IsHighRiskContract`, `new_monthly_customer` e `is_electronic_check`; uma faixa de tempo de casa para a apresentação; somas de serviços trocadas por `n_protecao` e `n_streaming`; fora `tenure_years`, `average_monthly_charges`, `IncomeCommitment` e `income_after_charges` | ◐ Sugestões no dicionário da ABT (04); seleção sugerida no 05 (D19) |

## 9. Gravação do dicionário no Unity Catalog

**O que faço:** junto numa tabela só tudo o que medi nas seções 3 a 8 e gravo no catálogo, ao lado das tabelas bronze.
Também comento cada coluna das tabelas bronze com a descrição e o veredito.

**Por que gravar:** o dicionário deixa de ser só um notebook e passa a ser um dado consultável por qualquer pessoa do grupo
(por SQL ou pela aba *Catalog*). Os próximos notebooks leem o veredito daqui, em vez de repetir a análise.

> **O que fica gravado**
>
> | Tabela | Conteúdo |
> |---|---|
> | `dicionario_dados` | Uma linha por coluna (23): descrição, tipo, domínio, **confirmação (verificado ou premissa)**, inválidos, tratamento, descritiva, drift, teste, IV, avaliação, **critério**, justificativa e potencial de derivadas |
> | `dicionario_categorias` | Uma linha por variável qualitativa × categoria (45): frequência no histórico e na inferência, taxa de churn e WoE |
>
> **WoE (Weight of Evidence)** é a parcela de cada categoria no IV: ln(% dos que saíram ÷ % dos que ficaram).
> Positivo quer dizer mais churn que a média; negativo, menos; perto de zero, igual à média (A.7).

In [ ]:
# 9.1 Monta o dicionário final: uma linha por coluna, juntando tudo o que foi medido nas seções 3 a 8
def dominio_texto(d) -> str:
    """Domínio esperado escrito por extenso (lista de categorias ou intervalo numérico)."""
    if isinstance(d, list):
        return " | ".join(d)
    if isinstance(d, tuple):
        lo, hi = d
        return f"{lo} a {hi}" if hi is not None else f"{lo} ou mais"
    return str(d)

TRATAMENTO = {                                     # decisões do grupo em 10/11 (checklist D1 a D17)
    "customerID": "Fora do modelo; fica na gold como chave do Prediction.csv (D14)",
    "gender": "Sugestão: descartar, IV abaixo de 0,02 (D15)",
    "MonthlyIncome": "Sugestão: descartar, IV abaixo de 0,02 e base externa sem garantia de precisão (D12)",
    "tenure": "Recuperar 'unknown' por TotalCharges ÷ MonthlyCharges, arredondado; mediana do histórico no caso sem os dois (D4)",
    "PhoneService": "Padronizar a grafia: yes → Yes, no → No (D1); sugestão: descartar, IV abaixo de 0,02 (D15)",
    "MultipleLines": "Sugestão: descartar, IV abaixo de 0,02 e repete PhoneService (D15)",
    "TotalCharges": "Preencher nulos com MonthlyCharges × tenure, 0 se tenure = 0 (D2, D3); indicador fl_total_charges_ausente (D5); sugestão: descartar, repete tenure (D16)",
    "MonthlyCharges": "Sugestão: manter, ou trocar pela mensalidade relativa ao pacote (D16)",
    "CustomerFeedback": "Trocar quebras de linha por espaço (D8); fora da ABT: vazamento (D9); texto em gold_churn_texto",
}
for _s in ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]:
    TRATAMENTO[_s] = "Sugestão: tem = 1, não tem = 0, junto com InternetService (D10); a ABT traz as duas versões"

q = qualidade.pivot(index="coluna", columns="base", values=["vazios", "fora_do_dominio", "exemplos"])
dic = meta.copy()
dic["dominio_esperado"] = dic["dominio_esperado"].apply(dominio_texto)
dic["invalidos_historico"] = (q[("vazios", "historico")] + q[("fora_do_dominio", "historico")]).astype(int)
dic["invalidos_inferencia"] = (q[("vazios", "inferencia")] + q[("fora_do_dominio", "inferencia")]).reindex(dic.index).astype(float)  # Churn não existe na inferência
dic["exemplos_invalidos"] = q[("exemplos", "historico")].replace("", None)
dic["tratamento"] = dic.index.map(TRATAMENTO)

# descritiva: quantitativas (5.2) e qualitativas (5.6)
dic = dic.join(desc_num[["media", "desvio_padrao", "minimo", "mediana", "maximo"]])
dic = dic.join(resumo_quali.set_index("variavel")[["n_categorias", "categoria_mais_comum", "pct_mais_comum"]])

# drift (5.9), associação com o churn (7.1) e avaliação (8.1)
dic = dic.join(drift.set_index("variavel")[["PSI", "estabilidade"]].rename(columns={"PSI": "psi_drift"}))
dic = dic.join(assoc.set_index("variavel")[["teste", "p_valor", "medida_efeito", "efeito", "IV", "poder_iv"]])
dic = dic.join(aval.set_index("variavel")[["avaliacao", "criterio", "justificativa", "potencial_derivada"]])

dic = dic.reset_index(names="coluna")
dic.insert(0, "ordem", range(1, len(dic) + 1))                   # mantém a ordem original das colunas
print(dic.shape)
display(dic)

In [ ]:
# 9.2 Dicionário de categorias: uma linha por variável × categoria, com a taxa de churn e o WoE de cada uma
linhas = []
for col in QUALI:
    h = hist_raw[col]
    i = inf_raw[col]
    for cat in sorted(set(h) | set(i), key=lambda c: (-int((h == c).sum()), c)):
        em = h == cat
        n1, n0 = int(y[em].sum()), int((1 - y[em]).sum())
        p1 = max(n1 / y.sum(), 1e-4)
        p0 = max(n0 / (1 - y).sum(), 1e-4)
        linhas.append({"variavel": col, "categoria": cat, "n_historico": int(em.sum()),
                       "pct_historico": 100 * em.mean(), "pct_inferencia": 100 * (i == cat).mean(),
                       "taxa_churn_pct": 100 * y[em].mean() if em.any() else np.nan,
                       "woe": float(np.log(p1 / p0)),                     # > 0: mais churn que a média
                       "categoria_valida": cat in meta.loc[col, "dominio_esperado"]})
dic_cat = pd.DataFrame(linhas)
print(dic_cat.shape)
display(dic_cat.round(2))

In [ ]:
# 9.3 Grava as duas tabelas no Unity Catalog e comenta tabelas e colunas
def para_spark(df: pd.DataFrame):
    """Converte para Spark trocando NaN por nulo nas colunas de texto."""
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype) in ("string", "str"):
            df[c] = df[c].astype(object).where(df[c].notna(), None)
    return spark.createDataFrame(df)

for nome, df in [("dicionario_dados", dic), ("dicionario_categorias", dic_cat)]:
    (para_spark(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.{nome}"))
    print(f"{FQ}.{nome}: {spark.table(f'{FQ}.{nome}').count()} linhas")

def sql_texto(t: str) -> str:
    return str(t).replace("'", "\\'")

# comentário de cada coluna das tabelas bronze, com a descrição e o veredito
for tabela in ["bronze_history", "bronze_inference"]:
    colunas = spark.table(f"{FQ}.{tabela}").columns
    for _, r in dic.iterrows():
        if r["coluna"] in colunas:
            premissa = " | PREMISSA, a confirmar" if str(r["confirmacao"]).startswith("PREMISSA") else ""
            txt = f"{r['descricao']} | {r['tipo_estatistico']} | avaliação: {r['avaliacao']}{premissa}"
            spark.sql(f"ALTER TABLE {FQ}.{tabela} ALTER COLUMN `{r['coluna']}` COMMENT '{sql_texto(txt)}'")

COMENTARIOS = {
    "dicionario_dados": "Dicionário de dados do case Terra Signal: uma linha por coluna, com tipo, qualidade, descritiva, drift, associação com o churn e avaliação",
    "dicionario_categorias": "Uma linha por variável qualitativa x categoria: frequência no histórico e na inferência, taxa de churn e WoE",
}
for nome, txt in COMENTARIOS.items():
    spark.sql(f"COMMENT ON TABLE {FQ}.{nome} IS '{sql_texto(txt)}'")
print("Comentários gravados nas tabelas bronze e no dicionário.")

In [ ]:
%sql
-- 9.4 Conferência: o dicionário lido direto do catálogo, como qualquer pessoa do grupo vai ver
SELECT ordem, coluna, grupo, tipo_estatistico, avaliacao, criterio, round(IV, 3) AS IV, poder_iv,
       invalidos_historico, tratamento, confirmacao, potencial_derivada
FROM workspace.terra_signal.dicionario_dados
ORDER BY ordem

> **Explicação**
>
> - O `dicionario_dados` tem uma linha por coluna da base (23): é o resumo de tudo o que fiz neste notebook. Para saber se
>   uma variável é boa, basta filtrar por `avaliacao`; para saber o que foi assumido, filtrar `confirmacao` por "PREMISSA".
> - O `dicionario_categorias` tem 45 linhas e mostra **onde** está o poder de cada variável. Exemplos:
>   - `Contract`: WoE de +0,71 no mensal contra −2,38 no bienal;
>   - `PaymentMethod`: só o "Electronic check" tem WoE positivo (+0,81); as outras três formas ficam abaixo da média;
>   - as categorias "no" e "yes" de `PhoneService` aparecem com `categoria_valida = false`, que é a grafia que vou corrigir.
> - **Um achado que só apareceu aqui:** em `StreamingTV` e `StreamingMovies`, quem tem e quem não tem o serviço sai quase na mesma
>   proporção (31% contra 33%; 30% contra 34%). O IV dessas variáveis vem quase todo da categoria "No internet service"
>   (7,2% de churn), ou seja, repete `InternetService`. Na criação de variáveis, vou testar se elas ainda acrescentam algo
>   depois de virarem "tem" ou "não tem" o serviço.
> - Comentei as colunas das tabelas bronze com descrição, tipo e veredito, com um aviso nas descrições que forem premissa (hoje, nenhuma).
>   Os comentários aparecem na aba *Catalog*, em `workspace.terra_signal.bronze_history`, aba *Columns*.

> **Anotações [André]: dúvidas para o grupo**
>
> 1. As tabelas do dicionário são sobrescritas a cada execução. Alguém vê problema, ou preferem guardar versões com data?
> 2. Os próximos notebooks vão ler o veredito da `dicionario_dados`. Todos conseguem consultar a tabela pelo SQL Editor?
>
> **Respostas e decisões do grupo:**
>
> &nbsp;

## 10. Valor do churn e linha de base

**O que faço:** calculo quanto o churn custa e quanto se consegue capturar **sem modelo**, com regras simples. É a
régua que o modelo precisa superar.

In [ ]:
# 10.1 Quanto vale um churn: a receita que sai com os clientes que cancelam
mc10 = pd.to_numeric(hist_raw["MonthlyCharges"], errors="coerce")
tn10 = pd.to_numeric(hist_raw["tenure"], errors="coerce")
valor = pd.DataFrame({
    "grupo": ["saiu", "ficou"],
    "clientes": [int(y.sum()), int((y == 0).sum())],
    "mensalidade_media": [mc10[y == 1].mean(), mc10[y == 0].mean()],
    "mensalidade_mediana": [mc10[y == 1].median(), mc10[y == 0].median()],
    "tempo_de_casa_medio": [tn10[y == 1].mean(), tn10[y == 0].mean()],
    "receita_mensal_total": [mc10[y == 1].sum(), mc10[y == 0].sum()],
})
valor["pct_da_receita"] = 100 * valor["receita_mensal_total"] / mc10.sum()
display(valor.round(2))

V_ANUAL = 12 * mc10[y == 1].mean()          # um ano de mensalidade de quem sai: valor de reter um cliente por um ano
print(f"Os {virg(y.sum(), 0)} clientes que saíram pagavam R$ {virg(mc10[y == 1].sum(), 0)} por mês, "
      f"{virg(valor.loc[0, 'pct_da_receita'], 1)}% da receita mensal do histórico.")
print(f"Cada cliente que sai leva, em média, R$ {virg(mc10[y == 1].mean(), 2)} por mês, "
      f"ou R$ {virg(V_ANUAL, 0)} em um ano.")

# a mesma conta para a inferência, antes de qualquer modelo: taxa de churn do contrato × mensalidade do cliente
mc_inf = pd.to_numeric(inf_raw["MonthlyCharges"], errors="coerce")
p_contrato = inf_raw["Contract"].map(y.groupby(hist_raw["Contract"]).mean())
print(f"Na inferência, a taxa do contrato prevê cerca de {virg(p_contrato.sum(), 0)} cancelamentos entre "
      f"{virg(len(inf_raw), 0)} clientes, com R$ {virg((p_contrato * mc_inf).sum(), 0)} de receita mensal em risco.")

> **Explicação (10.1): o valor de um churn**
>
> **O que calculei:** a receita mensal que os clientes que saíram levavam. É a parte da pergunta sobre o valor do churn que os
> dados respondem sozinhos. A outra parte, quanto custa a ação de retenção, depende do negócio (perguntas para os organizadores, no notebook 00).
>
> **O que encontrei:**
> - Os 1.504 clientes que saíram pagavam R$ 112.243 por mês: **30,8% da receita mensal** do histórico, embora
>   sejam 26,7% dos clientes. Quem sai paga mais: R$ 74,63 de mensalidade média, contra R$ 60,96 de quem fica.
> - **Um cliente que sai leva cerca de R$ 896 por ano** (12 × R$ 74,63). É o valor usado no notebook 00 para escolher o limiar.
> - Na inferência, só com a taxa de churn de cada contrato, espero cerca de **376 cancelamentos** e **R$ 24.931 de
>   receita mensal em risco**. O modelo deve concentrar essa receita em menos clientes acionados.
>
> **Premissas:** `MonthlyCharges` em R$ (moeda confirmada pelos organizadores) e um ano como horizonte de valor. Com outro horizonte,
> o valor muda na mesma proporção.

In [ ]:
# 10.2 A linha de base: quanto se captura sem modelo (sorteio, acionar todos e regras simples)
fibra10 = hist_raw["InternetService"].eq("Fiber optic")
mensal10 = hist_raw["Contract"].eq("Month-to-month")
cheque10 = hist_raw["PaymentMethod"].eq("Electronic check")
sem_suporte10 = hist_raw["InternetService"].ne("No") & hist_raw["TechSupport"].eq("No")
REGRAS = {
    "Acionar todos": pd.Series(True, index=hist_raw.index),
    "Contrato mensal": mensal10,
    "Internet sem suporte": sem_suporte10,
    "Cheque eletrônico": cheque10,
    "Fibra sem suporte": fibra10 & hist_raw["TechSupport"].eq("No"),
    "Mensal e cheque eletrônico": mensal10 & cheque10,
    "Até 6 meses e mensal": (tn10 <= 6) & mensal10,
}
TAXA = y.mean()
base10 = pd.DataFrame([{"regra": nome,
                        "pct_acionados": 100 * m.mean(),                     # parte da carteira que o time liga
                        "pct_churn_capturado": 100 * y[m].sum() / y.sum(),   # recall: dos que saem, quantos pego
                        "precisao_pct": 100 * y[m].mean(),                   # dos que ligo, quantos iam sair
                        "lift": y[m].mean() / TAXA}                          # quantas vezes melhor que o sorteio
                       for nome, m in REGRAS.items()]).sort_values("pct_acionados")
display(base10.round(2))
print(f"Sorteio: acionar qualquer x% da carteira captura x% dos que saem, com precisão de {virg(100 * TAXA, 1)}% (lift 1).")

In [ ]:
# 10.2b A linha de base em um gráfico: o modelo precisa ficar acima desses pontos
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.plot([0, 100], [0, 100], color=CINZA, linestyle="--", linewidth=1.2)
ax.text(72, 62, "linha do sorteio:\nacionar x% captura x%", color="#888888", fontsize=9)
DESL10 = {"Contrato mensal": (8, -4), "Internet sem suporte": (8, -12), "Cheque eletrônico": (8, -12),
          "Fibra sem suporte": (8, 6), "Mensal e cheque eletrônico": (-10, 4), "Até 6 meses e mensal": (8, -2),
          "Acionar todos": (-10, -16)}
for _, r in base10.iterrows():
    destaque = r["regra"] in ("Mensal e cheque eletrônico", "Até 6 meses e mensal")
    ax.scatter(r["pct_acionados"], r["pct_churn_capturado"], s=70, zorder=3,
               color=DESTAQUE if destaque else (CINZA if r["regra"] == "Acionar todos" else COR))
    dx, dy = DESL10[r["regra"]]
    ax.annotate(f"{r['regra']} (lift {virg(r['lift'], 2)})", (r["pct_acionados"], r["pct_churn_capturado"]),
                xytext=(dx, dy), textcoords="offset points", fontsize=8.5, color=TEXTO,
                ha="right" if dx < 0 else "left")
ax.set_xlim(0, 105)
ax.set_ylim(0, 105)
ax.set_xlabel("% da carteira acionada pelo time de vendas")
ax.set_ylabel("% dos clientes que saem capturados (recall)")
fig.suptitle("Sem modelo, regras simples já capturam o dobro do sorteio: acionando 26% da carteira\n"
             "(contrato mensal e cheque eletrônico), o time alcança 52% de quem vai sair",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "PaymentMethod", "InternetService", "TechSupport", "tenure"])
plt.show()

> **Como ler a tabela e o gráfico 10.2**
> - **Cada ponto é uma forma de escolher quem o time de vendas liga.** No eixo horizontal, quanto da carteira é acionado;
>   no vertical, quantos dos clientes que vão sair são alcançados (recall).
> - **A linha tracejada é o sorteio:** ligar para x% da carteira ao acaso alcança x% de quem sai. Quanto mais acima da
>   linha, melhor a regra.
> - **O lift** diz quantas vezes a regra acerta mais que o sorteio entre os acionados: lift 2 = metade dos acionados ia
>   sair, contra 26,7% no sorteio.
> - **O que procurar:** "mensal e cheque eletrônico" e "até 6 meses e mensal" (em laranja) dobram o sorteio acionando
>   um quinto a um quarto da carteira. É a régua para o modelo: ele precisa formar uma curva **acima** desses pontos.

> **Resumo da seção 10**
>
> Quem sai leva 30,8% da receita mensal e cerca de R$ 896 por ano cada. Sem modelo, regras simples já dobram o sorteio:
> "mensal e cheque eletrônico" pega 52% de quem sai acionando 26% da carteira. O modelo precisa formar uma curva acima
> dessas regras. A escolha da métrica e do limiar está no notebook 00.

## Resumo do notebook 03

> **Resumo do que fiz**
>
> | Etapa | O que fiz | O que encontrei |
> |---|---|---|
> | **1. Leitura** | Li os CSVs como texto, sem converter nada | **5.634** clientes no histórico e **1.409** na inferência. Os arquivos têm 5.912 e 1.461 linhas físicas porque o feedback tem quebras de linha dentro do texto |
> | **2. Bronze** | Gravei as bases como chegaram, com origem e data de carga | Tabelas `bronze_history` e `bronze_inference` no Unity Catalog |
> | **3. Metadados** | Descrevi cada coluna: grupo, tipo estatístico, papel, domínio esperado e o que foi verificado ou assumido | 23 colunas: 1 identificador, 20 explicativas, 1 texto livre e o alvo `Churn`. Moeda em R$. Nenhuma premissa em aberto: dúvidas respondidas pelos organizadores (10/11) |
> | **4. Qualidade** | Comparei cada valor com o domínio esperado e chequei regras entre colunas | 20 de 23 colunas sem problemas. Problemas só em `tenure` (19 `unknown`), `PhoneService` (grafia "yes"/"no") e `TotalCharges` (278 `null`). A ausência de `TotalCharges` se liga ao churn, e crio um indicador para ela |
> | **5. Descritiva** | Calculei medidas, histogramas, boxplots e frequências | Nenhum outlier pelo critério de Tukey. Distribuições com dois picos (novos e antigos; só telefonia e pacote com internet), o que justifica testes não paramétricos |
> | **5. Drift** | Comparei histórico e inferência (KS, qui-quadrado, PSI) | As 20 variáveis são estáveis (PSI máximo de 0,011). A inferência parece ter sido separada ao acaso da mesma base |
> | **6. Associação entre variáveis** | Medi os 190 pares com V de Cramér, Spearman e η | Cinco pares com repetição forte (≥ 0,8). A mensalidade é quase o tipo de internet (η = 0,91). A associação entre os serviços era só a categoria "No internet service" |
> | **7. Associação com o churn** | Medi teste, tamanho do efeito e IV de cada variável | Taxa geral de churn de **26,7%**. 12 variáveis com IV ≥ 0,30 e 4 sem poder preditivo. **O texto do feedback vaza o desfecho** |
> | **8. Avaliação** | Juntei tudo num veredito por variável, com critério, justificativa e ideias de derivadas | **13 boas, 3 com atenção, 6 para descartar**. As derivadas são testadas no notebook 04 |
> | **9. Gravação** | Gravei o dicionário no catálogo e comentei as colunas das tabelas bronze | Tabelas `dicionario_dados` (23 linhas) e `dicionario_categorias` (45 linhas) |
> | **10. Negócio** | Respondi às perguntas do grupo com os dados: valor do churn, linha de base e regra do limiar | Quem sai leva 30,8% da receita mensal (R$ 896 por ano cada). Regras simples dobram o sorteio. O limiar depende do custo da ação; 13 perguntas para os organizadores |
>
> **Os sete achados principais**
> 1. **A base é limpa.** Os problemas estão concentrados em três colunas e têm tratamento simples (seção 4).
> 2. **Treino e inferência têm o mesmo perfil.** O desempenho medido na validação deve se repetir nos clientes a pontuar,
>    e o churn esperado na inferência é de cerca de 27% (seção 5).
> 3. **O perfil de quem sai é claro:** contrato mensal (42,7% de churn contra 3,3% no bienal), primeiros meses de casa
>    (58,2% nos dois primeiros meses), fibra ótica, ausência de proteção e suporte e pagamento por cheque eletrônico (45,1%).
> 4. **Como o cliente se relaciona importa mais do que quem ele é.** Contrato e serviços estão no topo do IV; gênero e
>    renda não ajudam. E contrato, pagamento e serviços são coisas que a empresa pode mudar com uma oferta de retenção.
> 5. **Quatro variáveis não ajudam:** `gender`, `PhoneService`, `MonthlyIncome` e `MultipleLines` (IV < 0,02).
> 6. **O preço é quase todo o pacote.** A mensalidade é 82% explicada pelo tipo de internet (η² = 0,82) e repete os
>    serviços; `TotalCharges` repete `tenure`. A informação útil do preço é o que vai além do pacote.
> 7. **O texto do feedback é vazamento.** Uma regra de uma linha acerta 98,9% dos clientes (IV de 11,4 contra 1,17 do
>    `Contract`). O texto foi escrito depois do cancelamento e a partir do próprio registro do cliente: 88,7% citam a
>    mensalidade exata. Não entra no modelo de previsão (argumentos na 7.3d).

## Checklist final do 03

| # | Decisão | Status | Evidência neste notebook |
|---|---|---|---|
| D17 | Ler das tabelas `bronze_*` | ☑ Aplicado | Célula da fonte: 5.634 e 1.409 clientes |
| D9 | `CustomerFeedback` fora do modelo oficial e da ABT; modelo com texto como contraexemplo, a partir da `gold_churn_texto` | ☑ Decidido | 7.3: uma regra sobre o texto acerta 98,9% (IV 11,4); 7.3d: os argumentos |
| D10 | Serviços como sim/não; a falta de internet só em `InternetService` | ◐ Sugestão; as duas versões na ABT | 6: a associação entre os serviços cai de 0,71–0,77 para 0,04–0,43 sem o "No internet service" |
| D12 | Descartar `MonthlyIncome` | ◐ Sugestão; fica na ABT | 7 e 8: IV abaixo de 0,02; base externa (Carlos) |
| D15 | Descartar `gender`, `PhoneService` e `MultipleLines` | ◐ Sugestão; ficam na ABT | 7 e 8: IV abaixo de 0,02 |
| D16 | `MonthlyCharges` e `TotalCharges` | ◐ Sugestão no 04 | 6: repetem `InternetService` e `tenure`; 8: veredito "Atenção" |

◐ = sugestão nossa; a seleção final das variáveis é feita na modelagem (D18).

**Também registrado, sem decisão em aberto:** nenhum outlier (Tukey), distribuições não normais (por isso os testes
não paramétricos), nenhum drift entre treino e inferência (PSI máximo de 0,011).

## Apêndice: os testes usados

A teoria de cada medida, com um exemplo calculado nos dados.

### A.1 Qui-quadrado de independência e V de Cramér: duas qualitativas

**Pergunta:** saber a categoria de uma variável diz alguma coisa sobre a categoria da outra?

**Por que este teste:** variáveis qualitativas não têm média. O que dá para comparar são **contagens** numa tabela
cruzada. O qui-quadrado compara as contagens observadas com as que **deveriam aparecer se as duas variáveis fossem
independentes**.

> **Como funciona: exemplo com `Partner` × `Dependents`**
>
> **1. Contagem esperada sob H0** (independência): esperado = total da linha × total da coluna ÷ n
>
> | Tem parceiro(a)? | Sem dependentes: observado | esperado | Com dependentes: observado | esperado |
> |---|---|---|---|---|
> | No | 2.616 | 2.037,6 | 291 | 869,4 |
> | Yes | 1.333 | 1.911,4 | 1.394 | 815,6 |
>
> Se fossem independentes, cerca de 30% dos clientes com e sem parceiro teriam dependentes. Na prática, só 10% de quem
> não tem parceiro(a) tem dependentes, contra 51% de quem tem.
>
> **2. Estatística:** χ² = Σ (observado − esperado)² ÷ esperado, somando as quatro células. Aqui, χ² ≈ 1.132, com
> 1 grau de liberdade ((linhas − 1) × (colunas − 1)).
>
> **3. Hipóteses e decisão:** H0: as duas variáveis são independentes. Com 1 grau de liberdade, o χ² precisaria passar
> de **3,84** para rejeitar H0 a 5%. O nosso passa de mil: rejeito H0.
>
> **4. Pressuposto:** contagem esperada de pelo menos 5 em cada célula. A menor aqui é 816, então o teste vale.

**O problema do qui-quadrado sozinho:** ele cresce com o número de clientes. Com o dobro de clientes e as mesmas
proporções, o χ² dobra, e a relação não ficou mais forte. Por isso ele diz **se** há relação, mas não **quanto**.

> **V de Cramér: o tamanho do efeito**
>
> V = √( χ² ÷ ( n × (k − 1) ) ), em que n é o número de clientes e k é o menor número de categorias entre as duas variáveis.
>
> Dividir por n tira o efeito do tamanho da amostra, e o resultado fica entre 0 e 1:
>
> | V | Leitura |
> |---|---|
> | 0 | Independentes: saber uma não diz nada sobre a outra |
> | 0,1 | Associação fraca |
> | 0,3 | Moderada |
> | 0,5 ou mais | Forte |
> | 1 | Uma determina a outra |
>
> **Exemplo:** `Partner` × `Dependents` tem V = √(1.132 ÷ 5.634) = **0,45**: associação de moderada a forte. Quem tem
> parceiro(a) tende a ter dependentes, mas uma variável não determina a outra.

**Na seção 7**, uso o mesmo teste e o mesmo V para comparar cada qualitativa com o churn.

### A.2 Correlação de Spearman (ρ): duas quantitativas

**Pergunta:** quando uma variável aumenta, a outra tende a aumentar (ou diminuir)?

**Por que Spearman, e não Pearson:** a correlação de Pearson mede só relações em **linha reta** e é sensível à forma das
distribuições. A seção 5 mostrou distribuições com dois picos (`tenure`, `MonthlyCharges`) e assimétricas (`TotalCharges`).
O Spearman correlaciona as **posições** (postos) dos valores, e não os valores em si.

> **Como funciona**
>
> 1. **Ordeno os clientes** por cada variável e troco cada valor pela sua **posição** (posto) nessa fila.
>    Empates recebem o **posto médio**: se três clientes empatam nas posições 10, 11 e 12, os três recebem 11.
> 2. **Calculo a correlação de Pearson entre os postos** das duas variáveis. Isso é o ρ (rô) de Spearman.
>
> Por usar postos, o Spearman:
> - não exige normalidade;
> - capta qualquer relação que **só sobe** ou **só desce** (monotônica), mesmo que não seja uma reta.
>
> | ρ | Leitura |
> |---|---|
> | −1 | Quando uma sobe, a outra sempre desce |
> | 0 | Sem tendência |
> | +1 | Quando uma sobe, a outra sempre sobe |
>
> H0: ρ = 0 (não há tendência). Nesta seção uso o **módulo** (|ρ|), porque o interesse é a força, e não a direção.
>
> **Exemplo, `tenure` × `TotalCharges`:** ρ = **0,89**. O total cobrado cresce quase sempre com o tempo de casa, o que
> faz sentido: total ≈ mensalidade × meses. A correlação de Pearson nos mesmos dados dá 0,82. O Spearman é maior porque a
> relação é monotônica, mas não é uma reta: clientes com a mesma idade de casa pagam mensalidades diferentes.

**Na seção 7**, uso o Spearman de outra forma: para testar a **tendência** do churn ao longo das categorias ordenadas do
`Contract`.

### A.3 Razão de correlação (η): quantitativa × qualitativa

**Por que mais uma medida:** nem o V de Cramér (duas qualitativas) nem o Spearman (duas quantitativas) servem para pares
**mistos**, como `MonthlyCharges` × `InternetService`. São 64 dos 190 pares da base.

**Pergunta:** quanto da variação de uma variável quantitativa é explicada pelos grupos de uma qualitativa?

> **Como funciona**
>
> Divido a variação total da quantitativa em duas partes:
> - **entre os grupos:** o quanto as médias dos grupos se afastam da média geral;
> - **dentro dos grupos:** o quanto cada cliente se afasta da média do seu grupo.
>
> η² = variação **entre** os grupos ÷ variação **total**
>
> Em fórmulas: variação total = Σ (valor − média geral)²; variação entre = Σ tamanho do grupo × (média do grupo − média geral)².
>
> | η | Leitura |
> |---|---|
> | 0 | As médias dos grupos são iguais: a qualitativa não diz nada sobre a quantitativa |
> | 0,5 | Associação forte: os grupos diferem bastante |
> | 1 | Dentro de cada grupo, todos têm o mesmo valor: a qualitativa determina a quantitativa |
>
> **Exemplo, `MonthlyCharges` por `InternetService`:**
>
> | Tipo de internet | Mensalidade média | Desvio-padrão |
> |---|---|---|
> | No | R$ 21,1 | 2,2 |
> | DSL | R$ 57,9 | 16,3 |
> | Fiber optic | R$ 91,5 | 12,6 |
>
> As médias são muito diferentes, e a variação dentro de cada grupo é pequena. η = 0,91, ou seja, η² = 0,82: **82% da
> variação da mensalidade é explicada só pelo tipo de internet**.

**A ligação com o que já conheço:** η² é exatamente o R² de uma ANOVA de um fator, ou de uma regressão da quantitativa
sobre a qualitativa transformada em variáveis 0/1. É a decomposição da variância entre e dentro dos grupos.

**Cuidados:**
- A η não tem sinal. Diz **quanto** a qualitativa explica, não em que direção; a direção se lê pelas médias dos grupos.
- Ela é assimétrica: mede a quantitativa explicada pela qualitativa, e não o contrário. Para comparar pares, isso basta.
- Usa médias, então é sensível a valores extremos. Aqui isso não preocupa, porque a seção 5 não encontrou outliers.

**O que me diz para o negócio:** se η é alta, a quantitativa é, em boa parte, consequência da qualitativa. No exemplo,
a mensalidade é quase toda o reflexo do pacote contratado. O que sobra (pagar mais ou menos que a média do próprio tipo
de internet) pode ser a informação nova e útil.

### A.4 Qui-quadrado e V de Cramér contra o churn: exemplo com `Contract`

**Pergunta:** a proporção de churn é a mesma em todas as categorias da variável?

A teoria do qui-quadrado e do V de Cramér está na A.1. Aqui, a tabela cruzada é **categoria × churn** (saiu ou ficou).

> **Como funciona: exemplo com `Contract`**
>
> **1. Contagem esperada sob H0.** Se o contrato não tivesse relação com o churn, todos os contratos teriam os 26,7% de
> churn da carteira. A contagem esperada de cada célula é:
> esperado = total da linha × total da coluna ÷ n
>
> **2. Contribuição de cada célula:** (observado − esperado)² ÷ esperado. Quanto mais a célula foge do esperado, maior a parcela.
>
> | Contrato | Saíram: observado | Saíram: esperado | Contribuição | Ficaram: observado | Ficaram: esperado | Contribuição |
> |---|---|---|---|---|---|---|
> | Month-to-month | 1.322 | 827,3 | 295,8 | 1.777 | 2.271,7 | 107,7 |
> | One year | 138 | 317,1 | 101,2 | 1.050 | 870,9 | 36,8 |
> | Two year | 44 | 359,6 | 277,0 | 1.303 | 987,4 | 100,9 |
>
> **3. Estatística:** χ² = soma das seis contribuições = **919,4**, com 2 graus de liberdade
> ((linhas − 1) × (colunas − 1) = 2 × 1).
>
> **4. Hipóteses e decisão:** H0: a variável e o churn são independentes. H1: a proporção de churn muda entre as categorias.
> Com 2 graus de liberdade, o χ² precisaria passar de **5,99** para rejeitar H0 a 5% de significância. O nosso é mais de
> 150 vezes maior, e o p-valor é praticamente zero: rejeito H0.
>
> **5. Pressuposto:** contagem esperada de pelo menos 5 em cada célula. Aqui, a menor é 317, então o teste vale.
>
> **Repare:** o bienal contribui quase tanto quanto o mensal (277 contra 296, entre os que saíram). O χ² pesa tanto o
> **excesso** de saídas (mensal: 495 a mais que o esperado) quanto a **falta** delas (bienal: 316 a menos). Uma categoria
> "protetora" é tão informativa quanto uma "de risco", a mesma ideia que reaparece no IV (A.7).
>
> **6. V de Cramér:** como o churn tem duas categorias, k − 1 = 1 e V = √(χ² ÷ n) = √(919,4 ÷ 5.634) = **0,40**. Saber o
> contrato diz bastante sobre o risco de saída (42,7% no mensal contra 3,3% no bienal), mas não determina o resultado:
> há clientes mensais que ficam.

**O que me diz para o negócio:** o qui-quadrado responde "essa característica tem a ver com o cancelamento?", e o V
responde "quanto?". Uma variável com p-valor pequeno e V perto de zero (como `MultipleLines`, com V de 0,04) tem uma
relação real, mas pequena demais para ajudar a decidir quem ligar.

### A.5 Spearman contra o churn: a tendência na variável ordinal

**Pergunta:** à medida que o contrato fica mais longo (mensal → 1 ano → 2 anos), o churn cai de forma consistente?

**Por que este teste:** o qui-quadrado trata as categorias como se não tivessem ordem. Ele daria o mesmo resultado se eu
embaralhasse "mensal", "1 ano" e "2 anos". Para uma variável **ordinal**, a pergunta de negócio é mais específica: existe
uma **tendência**, um efeito de "dose" da fidelidade? O Spearman (A.2) usa a ordem, e o qui-quadrado não usa.

> **Como funciona aqui: muitos empates**
>
> Com só três categorias de um lado e duas do outro, quase todos os clientes empatam. Cada grupo recebe o **posto médio**
> (a média das posições que ocuparia na fila ordenada):
>
> | Contrato | Clientes | Posições na fila | Posto médio |
> |---|---|---|---|
> | Month-to-month | 3.099 | 1 a 3.099 | 1.550 |
> | One year | 1.188 | 3.100 a 4.287 | 3.693,5 |
> | Two year | 1.347 | 4.288 a 5.634 | 4.961 |
>
> O mesmo vale para o churn: os 4.130 que ficaram recebem um posto, e os 1.504 que saíram, outro. Depois, calculo a
> correlação de Pearson entre os postos.
>
> - Só a **ordem** importa: não preciso dizer que "1 ano" fica exatamente no meio entre "mensal" e "2 anos".
> - **Cuidado na leitura:** com tantos empates, o ρ não consegue chegar a ±1, mesmo com uma tendência perfeita. Por isso,
>   −0,40 é um valor alto neste contexto. O teste clássico para essa situação (tendência numa tabela ordinal × binária) é
>   o de Cochran-Armitage, e o Spearman leva à mesma conclusão.
>
> H0: ρ = 0 (não há tendência).

**Nos dados:** `Contract` tem **ρ = −0,40** (p < 0,001). O sinal negativo confirma a tendência que a tabela já
sugeria: 42,7% → 11,6% → 3,3% de churn conforme o contrato fica mais longo.

**O que me diz para o negócio:** cada degrau de fidelidade reduz o risco. Isso sustenta uma ação concreta: oferecer a
migração do contrato mensal para o anual, e não só tratar o mensal como um caso isolado.

### A.6 Mann-Whitney e correlação rank-bisserial

**Pergunta:** quem saiu tende a ter valores maiores (ou menores) da variável do que quem ficou?

**Por que este teste, e não o teste t:** o teste t compara **médias** e supõe distribuições parecidas com a normal. A
seção 5 mostrou que `tenure` e `MonthlyCharges` têm **dois picos** e que `TotalCharges` é assimétrica. Nesses casos, a
média representa mal o centro, e o teste t perde confiabilidade. O Mann-Whitney é o equivalente **não paramétrico**:
compara as **posições** (postos) dos valores, e não os valores em si.

> **Como funciona: a ideia dos pares**
>
> Imagine formar todos os pares possíveis (um cliente que saiu, um cliente que ficou). Em `tenure`, são
> 1.496 × 4.119 ≈ 6,2 milhões de pares. Em cada par, pergunto: quem tem mais tempo de casa?
>
> **U** = número de pares em que o cliente que saiu tem o valor maior, contando cada empate como meio par.
>
> | Situação | U ÷ total de pares |
> |---|---|
> | Os dois grupos são iguais (H0) | cerca de 0,50: metade dos pares para cada lado |
> | Quem saiu tem sempre valores menores | perto de 0 |
> | Quem saiu tem sempre valores maiores | perto de 1 |
>
> H0: um cliente sorteado entre os que saíram tem a mesma chance de ter valor maior ou menor que um sorteado entre os
> que ficaram. **Atenção:** o teste não compara medianas em geral. Ele só equivale a uma comparação de medianas quando
> as duas distribuições têm a mesma forma, o que não acontece aqui. Por isso, leio o resultado como "um grupo tende a ter
> valores maiores que o outro".

> **Correlação rank-bisserial: o tamanho do efeito**
>
> r = (% dos pares em que quem saiu tem valor maior) − (% dos pares em que tem valor menor)
>
> Vai de −1 a 1. Negativo: quem saiu tende a ter valores **menores**; positivo: **maiores**. Em módulo, a leitura é
> parecida com a de outras correlações: 0,1 pequeno, 0,3 médio, 0,5 grande. Na tabela 7.1 aparece o módulo (|r|); o
> sentido se lê pelas medianas, na coluna `observacao`.
>
> **Exemplo com `tenure`:**
>
> | Nos 6,2 milhões de pares, quem saiu tem... | % dos pares |
> |---|---|
> | mais tempo de casa | 25,6% |
> | o mesmo tempo (empate) | 2,0% |
> | menos tempo de casa | 72,4% |
>
> r = 25,6% − 72,4% ≈ **−0,47**: um efeito de moderado a grande, no sentido de quem saiu ser mais novo de casa.
> O U do teste é 25,6% + metade dos empates (1,0%) = 26,6% dos pares. As medianas confirmam: 37 meses entre quem ficou
> e 10 entre quem saiu.

**O que me diz para o negócio:** o r tem uma leitura direta. "Se eu pegar ao acaso um cliente que cancelou e um que
ficou, em quase 3 de cada 4 vezes o que cancelou é o mais novo de casa." É um argumento fácil de apresentar e mostra por
que os primeiros meses são o momento crítico da retenção.

### A.7 Information Value (IV) e Weight of Evidence (WoE)

**Pergunta:** quanto esta variável, sozinha, ajuda a **separar** quem sai de quem fica?

**Por que esta medida:** os testes anteriores têm escalas diferentes (V, ρ e r). Não dá para dizer se um V de 0,40
é "melhor" que um r de 0,47. O IV mede todas as variáveis, qualitativas e quantitativas, **na mesma escala**, e por isso
permite o ranking da 7.2. É a medida padrão em modelos de crédito e de churn. As quantitativas entram divididas em 10
faixas (decis).

> **Como funciona: exemplo com `Contract`**
>
> **1. Distribuo cada grupo pelas categorias:** que % dos que saíram está em cada contrato, e que % dos que ficaram.
>
> **2. WoE de cada categoria** = ln(% dos que saíram ÷ % dos que ficaram). Positivo: a categoria concentra mais quem
> sai; negativo, mais quem fica; zero, igual à média.
>
> **3. IV** = Σ (% dos que saíram − % dos que ficaram) × WoE.
>
> | Contrato | % dos que saíram | % dos que ficaram | WoE | Parcela do IV |
> |---|---|---|---|---|
> | Month-to-month | 87,9% | 43,0% | +0,71 | 0,321 |
> | One year | 9,2% | 25,4% | −1,02 | 0,166 |
> | Two year | 2,9% | 31,5% | −2,38 | 0,681 |
> | **IV** | | | | **1,17** |
>
> Repare que a maior parcela vem do bienal: quase ninguém que sai está nele. Uma categoria "protetora" separa tão bem
> quanto uma "de risco", a mesma ideia do qui-quadrado (A.4).
>
> **É a mesma conta do PSI (5.9a).** Lá, comparei a distribuição do histórico com a da inferência; aqui, a dos que saíram
> com a dos que ficaram. Nos dois casos, as parcelas são sempre positivas (diferença e logaritmo têm o mesmo sinal), e
> uma categoria pequena que muda muito em proporção pesa bastante.
>
> | IV | Poder preditivo |
> |---|---|
> | < 0,02 | Sem poder |
> | 0,02 a 0,10 | Fraco |
> | 0,10 a 0,30 | Médio |
> | 0,30 a 0,50 | Forte |
> | > 0,50 | Muito forte: **confiro se não é vazamento** |
>
> As faixas são uma convenção da literatura de *credit scoring*, e não uma regra estatística.

**A ligação com a regressão logística:** o WoE é o logaritmo da razão de chances da categoria em relação à média.
No mensal, a chance de sair é 1.322 ÷ 1.777 = 0,74, contra 1.504 ÷ 4.130 = 0,36 na carteira toda. A razão é 2,04, e
ln(2,04) = 0,71, exatamente o WoE. É a mesma linguagem (log da chance) que o modelo logístico vai usar.

**Por que "muito forte" pede verificação:** em bases reais de crédito, IV acima de 0,50 é raro. Quando aparece, a
primeira suspeita é a variável conter o próprio desfecho. Na 7.3, foi o caso do texto do feedback (IV de 11,4).
Contrato, tempo de casa e serviços também passam de 0,50, mas são informações que existem antes do cancelamento.

> **Limitações do IV**
> - **Depende das faixas nas quantitativas.** Com outros cortes (5 ou 20 faixas, em vez de 10), o valor muda um pouco.
>   Uso decis para todas, para que fiquem comparáveis entre si.
> - **Categoria sem ninguém que saiu** daria ln(0). O código usa um piso mínimo (0,0001) para evitar a divisão por zero.
> - **Olha uma variável por vez.** Não percebe que duas variáveis contam a mesma história: `TotalCharges` tem IV de 0,31,
>   mas repete `tenure` (seção 6). A seção 8 junta as duas informações.
> - **Mede separação, não causa.** Um IV alto diz que a variável distingue quem sai, não que ela provoca a saída.

**O que me diz para o negócio:** o IV ordena as variáveis pelo quanto separam quem sai de quem fica, e o WoE mostra,
dentro de cada variável, **qual categoria** concentra o risco. É dele que sai o perfil do cliente em risco. Para decidir
a **ação**, porém, ainda é preciso perguntar se a empresa consegue mudar aquela característica: dá para oferecer um
contrato anual, mas não dá para mudar a idade do cliente.